# Minor Project 02 (CA 03) — Exploratory Data Analysis
## Ordering Rhythm and Basket Behaviour in Instacart Grocery Orders

| | |
|---|---|
| **Course** | CSB314 — Exploratory Data Analysis |
| **Assessment** | Minor Project 02 / CA 03 (Unit IV + Unit V) |
| **Dataset** | Instacart Market Basket Analysis (Kaggle) — Option 2 of the assignment |
| **Source page** | https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis |
| **Student name / Roll no.** | *(write your name and roll number here)* |

**Central analytical question:** *How do customers' ordering rhythm (when they order and how long they wait between orders) and basket behaviour (basket size and share of re-ordered items) vary, and which variables are associated with each other?*

**How to read this notebook.** Every task follows the same pattern: a Markdown block *before* the code (**Objective → Concept → Methodology**), the code, its **actual output**, and a Markdown block *after* the output (**Interpretation → Observations → Decision → Conclusion**). All numbers quoted in the Markdown were produced by the code cells above them.

# 2. Problem Statement and Analytical Objectives

An online grocery marketplace wants to understand *when* customers shop, *how often* they come back and *what* their baskets look like, so that it can plan delivery capacity and design re-order reminders. We answer five concrete questions:

| # | Analytical question | Where answered |
|---|---|---|
| Q1 | When do customers order, and how long do they wait between orders? | Sections 8–10, 13 |
| Q2 | How large are baskets and how do they differ by day, time of day and customer? | Sections 7–10, 12 |
| Q3 | How much of a basket is re-ordered items, and which departments depend most on re-orders? | Section 11 |
| Q4 | Which variables are *associated* with the re-order rate (association only, no causal claims)? | Section 16 |
| Q5 | Are extremely large baskets errors or genuine behaviour, and how should they be treated? | Section 17 |

**Technical objectives (from the assignment):** feature engineering, `groupby/agg/transform`, `pivot_table/crosstab`, `melt/pivot/stack/unstack`, `merge/join/concat`, descriptive statistics, distribution shapes, empirical rule, sampling & CLT, covariance / Pearson / Spearman, IQR & Z-score outliers with removal / capping / retention, and a full data-quality assessment.

# 3. Dataset Source and Description

**Dataset:** *The Instacart Online Grocery Shopping Dataset 2017* — Kaggle mirror "Instacart Market Basket Analysis".
**Kaggle URL:** https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis
**Files used (all six):** `orders.csv`, `order_products__prior.csv`, `order_products__train.csv`, `products.csv`, `aisles.csv`, `departments.csv`.

| File | Grain (one row = …) | Primary key | Foreign keys | Key columns |
|---|---|---|---|---|
| `orders.csv` | one customer order | `order_id` | `user_id` | `order_number`, `order_dow`, `order_hour_of_day`, `days_since_prior_order`, `eval_set` |
| `order_products__prior.csv` | one product line inside a *prior* order | (`order_id`, `product_id`) | `order_id`, `product_id` | `add_to_cart_order`, `reordered` |
| `order_products__train.csv` | one product line inside a *train* order | (`order_id`, `product_id`) | `order_id`, `product_id` | `add_to_cart_order`, `reordered` |
| `products.csv` | one product | `product_id` | `aisle_id`, `department_id` | `product_name` |
| `aisles.csv` | one aisle | `aisle_id` | – | `aisle` |
| `departments.csv` | one department | `department_id` | – | `department` |

**Licence / usage conditions:** ⚠️ *Before submitting, open the Kaggle page above and copy the licence / terms text that is shown there into this cell* (for example: "Licence: ______ ; Usage: ______"). The assignment requires this to be recorded, and it must come from the live Kaggle page, not from memory.

**Important structural facts about this dataset** (verified in Section 5/6 with code):
* There is **no calendar timestamp**. Time information is `order_dow` (day of week, coded 0–6), `order_hour_of_day` (0–23) and `days_since_prior_order` (0–30). The assignment explicitly accepts this for Instacart.
* `eval_set` splits orders into `prior` (history), `train` (one labelled order per user) and `test` (one order per user whose items are **not** published). Orders in `test` therefore have **no** rows in the item files — this is an expected unmatched-record case we will study in Section 11.
* The day-of-week codes are **not labelled** in the files. We never assume names for the codes in core analysis; one clearly flagged *assumption* (codes 0 and 1 = weekend) is used for a single feature and is discussed in Section 7.

# 4. Importing Libraries and Loading Data

**Objective.** Import the tools, fix the random seed, and load the data in a way that fits in memory and is reproducible.

**Libraries.** `pandas`, `numpy`, `matplotlib`, `seaborn` are required by the assignment. **Additional library:** `scipy.stats`, used only for statistics that pandas does not provide directly (population skewness/kurtosis cross-check, the normal density curve / CDF, and `pearsonr`/`spearmanr`).

**Sampling decision (documented, reproducible).** The prior-order file alone has ~32 million rows (≈0.6 GB as CSV). The assignment allows a *documented, reproducible sample* for Instacart. We sample **users**, not rows: we draw 20,000 users at random (`numpy.random.default_rng(42)`) and keep **all** orders and order lines of those users. Sampling whole users keeps each customer's order history intact (needed for `days_since_prior_order`, order sequence and per-customer transforms) and keeps every table consistent with the others. Order-line files are read in chunks and filtered to the sampled orders so the full file never sits in memory.

**Data location.** Put the unzipped CSV files in a folder named `data/` next to this notebook (or directly next to it). No personal absolute paths are used.

In [1]:
import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats            # additional library: statistics helpers (skew/kurtosis, norm, pearsonr, spearmanr)

SEED = 42                          # fixed seed used for EVERY random operation in this notebook
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.3f}".format)

print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

pandas 3.0.2 | numpy 2.4.4 | seaborn 0.13.2


In [2]:
# ---- locate the files (relative paths only) ----
DATA_DIR = pathlib.Path("data") if pathlib.Path("data/orders.csv").exists() else pathlib.Path(".")
FILES = {
    "orders": "orders.csv", "prior": "order_products__prior.csv", "train": "order_products__train.csv",
    "products": "products.csv", "aisles": "aisles.csv", "departments": "departments.csv",
}
missing_files = [f for f in FILES.values() if not (DATA_DIR / f).exists()]
assert not missing_files, f"Place these CSV files in the data/ folder: {missing_files}"

# ---- load orders and draw a reproducible sample of users ----
SAMPLE_USERS = 20_000
orders_full = pd.read_csv(DATA_DIR / FILES["orders"])
all_users = np.sort(orders_full["user_id"].unique())
rng_sample = np.random.default_rng(SEED)
sampled_users = rng_sample.choice(all_users, size=SAMPLE_USERS, replace=False)

raw_orders = orders_full[orders_full["user_id"].isin(sampled_users)].reset_index(drop=True)   # RAW copy (never modified)
n_full_orders, n_full_users = len(orders_full), len(all_users)
del orders_full

# ---- load order lines in chunks, keeping only lines that belong to sampled orders ----
sampled_order_ids = set(raw_orders["order_id"])
LINE_DTYPES = {"order_id": "int32", "product_id": "int32", "add_to_cart_order": "int16", "reordered": "int8"}

def load_order_lines(filename):
    kept, rows_read = [], 0
    for chunk in pd.read_csv(DATA_DIR / filename, chunksize=2_000_000, dtype=LINE_DTYPES):
        rows_read += len(chunk)
        kept.append(chunk[chunk["order_id"].isin(sampled_order_ids)])
    return pd.concat(kept, ignore_index=True), rows_read

raw_prior, n_full_prior = load_order_lines(FILES["prior"])
raw_train, n_full_train = load_order_lines(FILES["train"])
raw_products    = pd.read_csv(DATA_DIR / FILES["products"])
raw_aisles      = pd.read_csv(DATA_DIR / FILES["aisles"])
raw_departments = pd.read_csv(DATA_DIR / FILES["departments"])

sample_summary = pd.DataFrame({
    "full dataset":  [n_full_users, n_full_orders, n_full_prior, n_full_train, len(raw_products), len(raw_aisles), len(raw_departments)],
    "our sample":    [SAMPLE_USERS, len(raw_orders), len(raw_prior), len(raw_train), len(raw_products), len(raw_aisles), len(raw_departments)],
}, index=["users", "orders", "prior order lines", "train order lines", "products", "aisles", "departments"])
sample_summary["sample share (%)"] = 100 * sample_summary["our sample"] / sample_summary["full dataset"]
sample_summary

,full dataset,our sample,sample share (%)
users,206209,20000,9.699
orders,3421083,329109,9.620
prior order lines,32434489,3101947,9.564
train order lines,1384617,135684,9.799
products,49688,49688,100.000
aisles,134,134,100.000
departments,21,21,100.000


**Interpretation.** The sample contains 20,000 customers (9.70% of 206,209), 329,109 orders (9.62%) and 3,237,631 order lines (3,101,947 prior + 135,684 train). Products, aisles and departments are small reference tables, so they are loaded in full (100%).
**Observations.** (1) The sampling share is almost the same for users, orders and lines (≈9.6–9.8%), which is what we expect when whole users are sampled — each customer's history stays complete. (2) The prior file really has about 32.4 million rows, which confirms why sampling was needed.
**Decision.** Keep the 20,000-user sample (seed 42) for all analysis; the `raw_*` frames are never modified.
**Conclusion.** Data is loaded reproducibly and every later number refers to this documented sample, not to all of Instacart.

# 5. Dataset Understanding

**Objective.** Understand each table's shape, column types and grain *before* any cleaning or joining.

**Concept — grain.** The *grain* of a table is what one row represents. Joining tables of different grain without care multiplies rows (Section 11). Here `orders` is one-row-per-order, but the two item files are one-row-per-product-in-an-order, so one order appears many times in them.

**Methodology.** Use `.shape`, `.dtypes`, `.head()` and `.nunique()` on every raw table. Raw tables are kept untouched as `raw_*`; all later work happens on copies.

In [3]:
raw_tables = {"orders": raw_orders, "order_products__prior": raw_prior, "order_products__train": raw_train,
              "products": raw_products, "aisles": raw_aisles, "departments": raw_departments}
overview = pd.DataFrame({
    "rows": {k: len(v) for k, v in raw_tables.items()},
    "columns": {k: v.shape[1] for k, v in raw_tables.items()},
    "column names": {k: ", ".join(v.columns) for k, v in raw_tables.items()},
})
overview

,rows,columns,column names
orders,329109,7,"order_id, user_id, eval_set, order_number, ord..."
order_products__prior,3101947,4,"order_id, product_id, add_to_cart_order, reord..."
order_products__train,135684,4,"order_id, product_id, add_to_cart_order, reord..."
products,49688,4,"product_id, product_name, aisle_id, department_id"
aisles,134,2,"aisle_id, aisle"
departments,21,2,"department_id, department"


In [4]:
print("orders.dtypes:\n", raw_orders.dtypes.to_string(), "\n")
display(raw_orders.head())
print("order lines (prior) head and products head:")
display(raw_prior.head())
display(raw_products.head())

orders.dtypes:
 order_id                    int64
user_id                     int64
eval_set                      str
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64 



,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.000
2,473747,1,prior,3,3,12,21.000
3,2254736,1,prior,4,4,7,29.000
4,431534,1,prior,5,4,15,28.000


order lines (prior) head and products head:


,order_id,product_id,add_to_cart_order,reordered
0,25,9755,1,1
1,25,31487,2,0
2,25,37510,3,1
3,25,14576,4,1
4,25,22105,5,0


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [5]:
# eval_set meaning and how many orders/users each set has; orders per user
display(raw_orders["eval_set"].value_counts().rename("orders").to_frame())
orders_per_user = raw_orders.groupby("user_id").size()
print("orders per user -> min:", orders_per_user.min(), "| median:", orders_per_user.median(), "| max:", orders_per_user.max())
print("distinct values: dow", sorted(int(d) for d in raw_orders['order_dow'].unique()), "| hour", raw_orders['order_hour_of_day'].min(), "to", raw_orders['order_hour_of_day'].max(),
      "| days_since", raw_orders['days_since_prior_order'].min(), "to", raw_orders['days_since_prior_order'].max())

,orders
eval_set,
prior,309109
train,12798
test,7202


orders per user -> min: 4 | median: 10.0 | max: 100
distinct values: dow [0, 1, 2, 3, 4, 5, 6] | hour 0 to 23 | days_since 0.0 to 30.0


### 5.1 Combining the two order-line files into one working table

`order_products__prior` and `order_products__train` have an *identical schema* and describe the same thing (product lines inside orders) for different `eval_set` values. We stack them with `pd.concat` into a **working copy** `items_all`, adding a column that records the file each row came from so no information is lost. (Section 11 revisits `concat` and proves the two files do not overlap.) The raw frames stay untouched.

In [6]:
items_all = pd.concat([raw_prior.assign(eval_set="prior"), raw_train.assign(eval_set="train")], ignore_index=True)
print("rows prior:", len(raw_prior), "| rows train:", len(raw_train), "| rows items_all:", len(items_all))
print("row counts add up:", len(items_all) == len(raw_prior) + len(raw_train))
items_all.head()

rows prior: 3101947 | rows train: 135684 | rows items_all: 3237631
row counts add up: True


,order_id,product_id,add_to_cart_order,reordered,eval_set
0,25,9755,1,1,prior
1,25,31487,2,0,prior
2,25,37510,3,1,prior
3,25,14576,4,1,prior
4,25,22105,5,0,prior


# 6. Data Quality Assessment

**Objective.** Find and quantify real data-quality issues *before* analysing, and decide for each one whether to correct, exclude (for a specific calculation) or retain.

**Concept.** Data quality has several dimensions: **completeness** (missing values), **consistency** (same thing recorded the same way), **accuracy/plausibility** (values that are possible in the real world), **validity** (correct types and allowed values), **uniqueness** (duplicates) and **integrity** (keys that point to something that exists). Each can silently distort aggregates and statistics if ignored.

**Methodology.** One focused check per cell, always on the raw frames or a copy. No row is dropped here — we only *measure*, then record a decision in an issue log at the end of the section.

### 6.1 Completeness — missing values by column
**Objective:** count missing values per column and per table, with percentages, and understand *why* they are missing.
**Concept:** a missing value can be *structural* (it cannot exist by definition), *accidental*, or *disguised* (a placeholder such as the text "missing").
**Methodology:** `isna().sum()` and percentage; then test whether missingness in `days_since_prior_order` is fully explained by another column.

In [7]:
missing_report = pd.concat(
    {name: pd.DataFrame({"missing": df.isna().sum(), "missing_%": 100 * df.isna().mean()}) for name, df in raw_tables.items()},
    names=["table", "column"])
missing_report[missing_report["missing"] > 0] if (missing_report["missing"] > 0).any() else missing_report.head(0)

,,missing,missing_%
table,column,,
orders,days_since_prior_order,20000,6.077


In [8]:
# Is the missingness of days_since_prior_order explained by "first order of the user"?
first_order = raw_orders["order_number"] == 1
gap_missing = raw_orders["days_since_prior_order"].isna()
print("missing gaps:", gap_missing.sum(), "| first orders:", first_order.sum())
print("missing gap  <=>  order_number == 1 for every row:", bool((gap_missing == first_order).all()))
print("Orders per user whose order_number sequence is 1..n with no gaps:",
      bool((raw_orders.groupby('user_id')['order_number'].agg(['max','size']).eval('max == size')).all()))

missing gaps: 20000 | first orders: 20000
missing gap  <=>  order_number == 1 for every row: True
Orders per user whose order_number sequence is 1..n with no gaps: True


**Interpretation.** Exactly one column has missing values: `days_since_prior_order`, 20,000 values = 6.08% of orders. The check shows missing ⇔ `order_number == 1` for every single row, and every customer's order sequence runs 1…n without gaps.
**Observations.** (1) The missingness is **structural**: a first order has no previous order, so "days since prior order" cannot exist. (2) The number of missing values equals the number of sampled customers (20,000) — one first order each.
**Decision.** Keep these as NaN. Filling them with 0 would wrongly say "ordered again the same day".
**Conclusion.** Completeness is high; the only gaps are explainable, so gap-based statistics use the 309,109 orders that have a previous order.

### 6.2 Consistency — disguised missing labels, spelling and case
**Objective:** look for categories that *say* "missing/other", whitespace/case variants of the same text, and repeated labels.
**Concept:** a placeholder string like `"missing"` is **not** seen by `isna()` but behaves like missing data; it would pass silently into group-by results.
**Methodology:** inspect the aisle/department label tables, count products attached to placeholder categories, and compare `product_name` with a normalised version (`strip().lower()`).

In [9]:
print("aisles labelled 'missing':      ", raw_aisles[raw_aisles["aisle"].str.lower().isin(["missing"])].to_dict("records"))
print("departments labelled 'missing': ", raw_departments[raw_departments["department"].str.lower().isin(["missing"])].to_dict("records"))
print("departments labelled 'other':   ", raw_departments[raw_departments["department"].str.lower().isin(["other"])].to_dict("records"))

missing_aisle_id = int(raw_aisles.loc[raw_aisles["aisle"].str.lower() == "missing", "aisle_id"].iloc[0])
missing_dept_id  = int(raw_departments.loc[raw_departments["department"].str.lower() == "missing", "department_id"].iloc[0])
n_prod_missing_aisle = int((raw_products["aisle_id"] == missing_aisle_id).sum())
n_prod_missing_dept  = int((raw_products["department_id"] == missing_dept_id).sum())
print(f"\nproducts in the 'missing' aisle: {n_prod_missing_aisle} ({100*n_prod_missing_aisle/len(raw_products):.2f}% of products)")
print(f"products in the 'missing' department: {n_prod_missing_dept} ({100*n_prod_missing_dept/len(raw_products):.2f}% of products)")

# spelling / case / whitespace variants of product names
name_raw  = raw_products["product_name"]
name_norm = name_raw.str.strip().str.lower()
print("\nnames with leading/trailing spaces:", int((name_raw != name_raw.str.strip()).sum()))
print("names that differ only by case/space (duplicates after normalising):", int(name_norm.duplicated().sum()), "| exact duplicate names:", int(name_raw.duplicated().sum()))
print("eval_set categories:", sorted(raw_orders['eval_set'].unique()))

aisles labelled 'missing':       [{'aisle_id': 100, 'aisle': 'missing'}]
departments labelled 'missing':  [{'department_id': 21, 'department': 'missing'}]
departments labelled 'other':    [{'department_id': 2, 'department': 'other'}]

products in the 'missing' aisle: 1258 (2.53% of products)
products in the 'missing' department: 1258 (2.53% of products)

names with leading/trailing spaces: 4
names that differ only by case/space (duplicates after normalising): 100 | exact duplicate names: 0
eval_set categories: ['prior', 'test', 'train']


**Interpretation.** Aisle 100 and department 21 are both literally labelled `missing`, and **1,258 products (2.53%)** sit in them — a *disguised* missing value that `isna()` cannot see. Department `other` (id 2) is a genuine catch-all label. Four product names carry leading/trailing spaces, and normalising case/spacing reveals 100 more names that collide with another product's name. There are no exact duplicate names.
**Observations.** (1) The "missing" aisle and department contain the same 1,258 products, so the placeholder is applied consistently. (2) `eval_set` has only the three valid labels (prior, test, train).
**Decision.** Keep the placeholder category but flag it (shown in red in the department chart; excluded when naming "lowest" categories).
**Conclusion.** Text consistency is acceptable; the placeholder category must be treated as *unknown category*, not a real department.

In [10]:
# Investigate (do not blindly drop): products whose names collide after normalising case/whitespace
norm_name = raw_products["product_name"].str.strip().str.lower()
near_dup = raw_products.assign(norm_name=norm_name)[norm_name.duplicated(keep=False)].sort_values("norm_name")
grp = near_dup.groupby("norm_name").agg(variants=("product_id", "size"), n_aisles=("aisle_id", "nunique"), n_departments=("department_id", "nunique"))
print("products involved:", len(near_dup), "| name groups:", len(grp), "| extra (redundant-looking) product_ids:", int((grp['variants'] - 1).sum()))
print("groups whose variants sit in different aisles:", int((grp['n_aisles'] > 1).sum()), "| in different departments:", int((grp['n_departments'] > 1).sum()))
near_dup.head(8)

products involved: 199 | name groups: 99 | extra (redundant-looking) product_ids: 100
groups whose variants sit in different aisles: 19 | in different departments: 13


,product_id,product_name,aisle_id,department_id,norm_name
31844,31845,18-In-1 Hemp Peppermint Pure-Castile Soap,25,11,18-in-1 hemp peppermint pure-castile soap
23339,23340,18-in-1 Hemp Peppermint Pure-Castile Soap,25,11,18-in-1 hemp peppermint pure-castile soap
19941,19942,Aged Balsamic Vinegar Of Modena,19,13,aged balsamic vinegar of modena
13152,13153,Aged Balsamic Vinegar of Modena,19,13,aged balsamic vinegar of modena
24830,24831,Albacore Solid White Tuna in Water,95,15,albacore solid white tuna in water
22582,22583,Albacore Solid White Tuna In Water,95,15,albacore solid white tuna in water
515,516,American Cheese Slices,21,16,american cheese slices
9037,9038,American Cheese slices,21,16,american cheese slices


**Interpretation.** 199 products form 99 groups whose names differ only by capitalisation or spacing (e.g. "Bag of Oranges" vs "Bag Of Oranges"), i.e. 100 extra product IDs. Most groups sit in the same aisle, but 19 groups span different aisles and 13 span different departments.
**Observations.** (1) These are different `product_id`s, so they are distinct catalogue entries rather than duplicate rows. (2) Because some variants sit in different aisles/departments, we cannot assume they are the same item.
**Decision.** Do **not** merge or drop them — merging would be a guess. We accept that product-level rankings may split the demand for one real item across IDs.
**Conclusion.** The issue affects product-level counts slightly but not department-level results.

### 6.3 Accuracy / plausibility — domain rules
**Objective:** test values against real-world rules.
**Concept:** a value can be the right *type* yet impossible (e.g. hour = 27, negative quantities). Rules used here: `order_dow` ∈ 0–6; `order_hour_of_day` ∈ 0–23; `days_since_prior_order` ∈ 0–30; `order_number` ≥ 1; `add_to_cart_order` ≥ 1 and forms the sequence 1…n inside each order; `reordered` ∈ {0,1}; and a **logical** rule — *a customer's very first order cannot contain re-ordered items* (nothing was bought before).
**Methodology:** boolean masks counted with `.sum()`, plus a group-level check of the cart-position sequence.

In [11]:
rules = {
    "order_dow outside 0-6":               ~raw_orders["order_dow"].between(0, 6),
    "order_hour_of_day outside 0-23":      ~raw_orders["order_hour_of_day"].between(0, 23),
    "days_since_prior_order outside 0-30": ~raw_orders["days_since_prior_order"].dropna().between(0, 30),
    "order_number < 1":                    raw_orders["order_number"] < 1,
    "add_to_cart_order < 1":               items_all["add_to_cart_order"] < 1,
    "reordered not in {0,1}":              ~items_all["reordered"].isin([0, 1]),
}
plaus = pd.Series({k: int(v.sum()) for k, v in rules.items()}, name="violations").to_frame()

# cart-position sequence 1..n inside every order
seq = items_all.groupby("order_id")["add_to_cart_order"].agg(["size", "min", "max"])
plaus.loc["orders whose cart positions are not exactly 1..n"] = int(((seq["min"] != 1) | (seq["max"] != seq["size"])).sum())

# logical rule: first orders must have zero re-ordered items
order_number_of = raw_orders.set_index("order_id")["order_number"]
first_order_lines = items_all[items_all["order_id"].map(order_number_of) == 1]
plaus.loc["re-ordered lines inside a user's FIRST order"] = int(first_order_lines["reordered"].sum())
plaus["checked rows"] = [len(raw_orders), len(raw_orders), int(raw_orders['days_since_prior_order'].notna().sum()), len(raw_orders),
                         len(items_all), len(items_all), len(seq), len(first_order_lines)]
plaus

,violations,checked rows
order_dow outside 0-6,0,329109
order_hour_of_day outside 0-23,0,329109
days_since_prior_order outside 0-30,0,309109
order_number < 1,0,329109
add_to_cart_order < 1,0,3237631
"reordered not in {0,1}",0,3237631
orders whose cart positions are not exactly 1..n,0,321907
re-ordered lines inside a user's FIRST order,0,199357


**Interpretation.** Every rule has 0 violations: day, hour and gap are inside their allowed ranges; order numbers and cart positions start at 1; `reordered` is only 0/1; every one of the 321,907 orders has cart positions exactly 1…n; and none of the 199,357 lines in first orders is flagged as a re-order.
**Observations.** (1) The logical rule (no re-orders in a first order) is a strong integrity test and it passes. (2) The data therefore needs no correction for impossible values.
**Decision.** No records are excluded for implausibility.
**Conclusion.** Accuracy/plausibility is good for this dataset.

In [12]:
# days_since_prior_order is capped at 30: how many orders sit exactly on the cap?
gap = raw_orders["days_since_prior_order"].dropna()
print("share of orders with gap == 30:", f"{100*(gap == 30).mean():.2f}%", "| count:", int((gap == 30).sum()))
print("share with gap == 29 / 28 / 27 for comparison:", [f"{100*(gap == d).mean():.2f}%" for d in (29, 28, 27)])
print("all gap values whole numbers:", bool((gap % 1 == 0).all()))

share of orders with gap == 30: 11.52% | count: 35595
share with gap == 29 / 28 / 27 for comparison: ['0.63%', '0.83%', '0.69%']
all gap values whole numbers: True


**Interpretation.** 35,595 orders (11.52% of orders with a previous order) have a gap of exactly 30 days, compared with only 0.63%, 0.83% and 0.69% at 29, 28 and 27 days. All values are whole numbers.
**Observations.** (1) The jump from <1% to 11.5% at 30 shows the value is a **cap**: any gap of 30 days *or more* is stored as 30 (right-censoring). (2) Therefore the mean gap (11.14) *under-estimates* the true average wait.
**Decision.** Retain the values, but read 30 as "30 or more"; the feature `gap_category` has an explicit "30+ days (capped)" band.
**Conclusion.** This is a validity limitation of the source, important for the bimodal discussion in Section 13.

### 6.4 Duplicates, validity / data types and key integrity
**Objective:** check exact duplicate rows, duplicate primary keys, data types, and whether every foreign key points at an existing parent row.
**Concept:** *primary key* = column(s) that must be unique per row; *foreign key* = column that must exist in the parent table. A duplicate key may be a valid repeated event or an error — it must be investigated, not silently dropped.
**Methodology:** `duplicated()`, `is_unique`, and `isin()` / anti-join style checks (the formal merge-indicator version is in Section 11).

In [ ]:
dup_report = pd.DataFrame({
    "exact duplicate rows": {
        "orders": raw_orders.duplicated().sum(), "items_all (prior+train)": items_all.duplicated().sum(),
        "products": raw_products.duplicated().sum(), "aisles": raw_aisles.duplicated().sum(), "departments": raw_departments.duplicated().sum()},
    "duplicate primary keys": {
        "orders": raw_orders["order_id"].duplicated().sum(),
        "items_all (prior+train)": items_all.duplicated(["order_id", "product_id"]).sum(),
        "products": raw_products["product_id"].duplicated().sum(),
        "aisles": raw_aisles["aisle_id"].duplicated().sum(), "departments": raw_departments["department_id"].duplicated().sum()},
})
dup_report

**Interpretation.** There are no exact duplicate rows and no duplicate primary keys in any table, including the composite key (`order_id`, `product_id`) of the order lines.
**Observations.** (1) A product appears at most once per order, so quantities are not hidden in repeated rows. (2) Nothing needs to be dropped.
**Decision.** No de-duplication.
**Conclusion.** Uniqueness holds, so counts of order lines are real counts of distinct products in orders.

In [ ]:
# Key integrity: do foreign keys exist in the parent table?
fk_checks = pd.DataFrame([
    ("items_all.order_id -> orders.order_id",           ~items_all["order_id"].isin(raw_orders["order_id"])),
    ("items_all.product_id -> products.product_id",     ~items_all["product_id"].isin(raw_products["product_id"])),
    ("products.aisle_id -> aisles.aisle_id",            ~raw_products["aisle_id"].isin(raw_aisles["aisle_id"])),
    ("products.department_id -> departments.department_id", ~raw_products["department_id"].isin(raw_departments["department_id"])),
], columns=["foreign key -> parent key", "orphan mask"])
fk_checks["child rows"]    = [len(items_all), len(items_all), len(raw_products), len(raw_products)]
fk_checks["orphan rows"]   = fk_checks["orphan mask"].apply(lambda m: int(m.sum()))
fk_checks = fk_checks.drop(columns="orphan mask")

# reverse direction: parents that have no children (not an error, but must be understood)
orders_without_items = raw_orders[~raw_orders["order_id"].isin(items_all["order_id"])]
print("orders with NO order lines:", len(orders_without_items), "-> by eval_set:", orders_without_items["eval_set"].value_counts().to_dict())
print("prior/train order ids that overlap:", len(set(raw_prior['order_id']) & set(raw_train['order_id'])))
fk_checks

**Interpretation.** All four foreign keys resolve with 0 orphan rows. In the other direction, 7,202 orders have no order lines — and all of them are `eval_set = "test"`. The prior and train line files share no order ids.
**Observations.** (1) Orphans in the child→parent direction would be errors; here there are none. (2) Parents without children (test orders) are **expected**, because Instacart withheld test-order contents.
**Decision.** Retain the test orders in `orders` (their basket features stay NaN); basket analysis uses the 321,907 orders that have lines.
**Conclusion.** Key integrity is perfect, and the one unmatched group is explained by how the dataset was published.

In [ ]:
dtype_report = pd.concat({n: df.dtypes.astype(str) for n, df in raw_tables.items()}).unstack(0)
dtype_report

### 6.5 Data-quality issue log (decision for every issue)
**Objective:** summarise every *actual* finding with its size, the decision taken and its effect on later analysis.
**Methodology:** the table is assembled by code from the numbers computed above (no hand-typed counts).

In [ ]:
n_first = int(first_order.sum())
issue_log = pd.DataFrame([
    ("Completeness", "days_since_prior_order missing", f"{int(gap_missing.sum()):,} orders ({100*gap_missing.mean():.2f}%) — all are each user's first order",
     "RETAIN as NaN (structural, not an error); never fill with 0", "Means/medians of the gap use only orders that have a previous order"),
    ("Completeness", "orders with no order lines", f"{len(orders_without_items):,} orders, eval_set = {sorted(orders_without_items['eval_set'].unique())}",
     "RETAIN; item-based features stay NaN for them", "Basket statistics describe only prior+train orders"),
    ("Consistency", "disguised missing category 'missing'", f"{n_prod_missing_aisle:,} products in aisle 'missing', {n_prod_missing_dept:,} in department 'missing'",
     "RETAIN but flag; exclude from department rankings when naming categories", "Would appear as a fake category in group-by results"),
    ("Consistency", "product names that differ only by case/space", f"{int((grp['variants'] - 1).sum())} extra product_ids in {len(grp)} name groups; {int((raw_products['product_name'] != raw_products['product_name'].str.strip()).sum())} names have leading/trailing spaces",
     "RETAIN all product_ids (distinct catalogue entries); do NOT merge or drop", "Product-level rankings could split one real product's demand across IDs; department-level results unaffected"),
    ("Validity", "days_since_prior_order capped at 30", f"{int((gap == 30).sum()):,} orders sit exactly at 30 ({100*(gap == 30).mean():.2f}%)",
     "RETAIN; treat 30 as '30 or more' (right-censored)", "Mean gap is under-estimated; creates an artificial spike"),
    ("Accuracy", "range / logic rules", f"{int(plaus['violations'].sum())} total violations across all rules",
     "No correction needed", "Values can be used as they are"),
    ("Duplicates", "exact duplicate rows / duplicate primary keys", f"{int(dup_report.sum().sum())} found",
     "Nothing to drop", "No double counting from duplicates"),
    ("Key integrity", "orphan foreign keys", f"{int(fk_checks['orphan rows'].sum())} orphan rows",
     "No action", "All joins to parent tables will be lossless"),
], columns=["Dimension", "Issue", "Size (computed)", "Decision", "Effect if ignored"])
issue_log

**Interpretation.** The issue log lists the real findings with computed sizes: 20,000 structural missing gaps, 7,202 test orders without lines, 1,258 products in the "missing" category, 100 case-variant product names, 35,595 capped gaps, and 0 problems in ranges, duplicates and keys.
**Observations.** (1) Most "issues" are properties of how the dataset was built, not errors — so the correct decision is mostly *retain and interpret*, not delete. (2) The two that most affect statistics are the **30-day cap** and the **structural NaNs**.
**Decision.** No rows are dropped in cleaning; every treatment is applied only inside a specific calculation.
**Conclusion.** The data is of high quality; the limitations above are carried into the analysis and the final conclusion.

# 7. Feature Engineering (Task 1)

**Objective.** Create analytical variables that make ordering rhythm and basket behaviour easier to study — at least four, using arithmetic, logical, conditional, string and aggregation logic.

**Concept.** A *feature* is a new column derived from existing ones so that a pattern becomes visible (e.g. a raw hour 0–23 becomes a readable "time of day" band). A good feature has a stated formula, a purpose, a known data type and an explicit rule for missing source values.

**Methodology.** Work on a copy (`orders = raw_orders.copy()`). Before each calculation the source values were already validated in Section 6 (no impossible values, no zero-length baskets). Missing source values are **preserved as NaN** (never filled with 0), because Section 6 showed they are structural.

**Assumption flagged for the viva.** The files do not say what `order_dow` 0–6 mean. The two busiest days are codes 0 and 1 (checked below), which is commonly read as the weekend, but this is an **assumption**. We therefore name days neutrally ("Day 0"…"Day 6") and use the weekend reading for **one** flag only, labelled `is_weekend_assumed`.

In [ ]:
orders = raw_orders.copy()                                   # WORKING copy; raw_orders stays untouched

# --- Feature 1 (conditional / binning): time_of_day from order_hour_of_day ---
orders["time_of_day"] = pd.cut(orders["order_hour_of_day"], bins=[-1, 5, 11, 17, 23],
                               labels=["Night (0-5)", "Morning (6-11)", "Afternoon (12-17)", "Evening (18-23)"])

# --- Feature 2 (conditional / binning): gap_category from days_since_prior_order (NaN stays NaN) ---
orders["gap_category"] = pd.cut(orders["days_since_prior_order"], bins=[-1, 0, 7, 14, 29, 30],
                                labels=["Same day (0)", "1-7 days", "8-14 days", "15-29 days", "30+ days (capped)"])

# --- Feature 3 (logical): is_first_order ---
orders["is_first_order"] = orders["order_number"] == 1

# --- Feature 4 (logical, ASSUMPTION): weekend flag, codes 0 and 1 ---
orders["is_weekend_assumed"] = orders["order_dow"].isin([0, 1])

# --- neutral, ordered day label used for grouping/plots ---
day_order = [f"Day {d}" for d in range(7)]
orders["dow_name"] = pd.Categorical([f"Day {d}" for d in orders["order_dow"]], categories=day_order, ordered=True)

# evidence for the weekend assumption: orders per day code
orders["dow_name"].value_counts().sort_index().rename("orders").to_frame().T

**Interpretation.** The order counts by day code are 57,764 (Day 0), 56,613 (Day 1) and between 40,663 and 44,873 for Days 2–6. The two highest codes are 0 and 1.
**Observations.** (1) The jump between codes 1 and 2 is the only large step, which is consistent with a weekend peak — but the source does not state the day names, so this is a **reading, not a fact**. (2) For that reason only one flag (`is_weekend_assumed`) depends on it.
**Decision.** Keep neutral labels ("Day 0"…"Day 6") for grouping and plots and flag the weekend feature as an assumption (to be stated in the viva).
**Conclusion.** Time-based features are built from `order_hour_of_day`, `days_since_prior_order` and the day code, which is as much as this dataset supports.

### 7.1 Item-based features (mathematical): `cart_size`, `reordered_items`, `reorder_rate`
`cart_size` = number of order lines in the order (count of products); `reordered_items` = sum of the 0/1 `reordered` flag; `reorder_rate` = `reordered_items / cart_size` (0–1). These come from the *child* table `items_all`, so we first **aggregate the child table to order grain** and only then attach it to `orders` — this avoids row multiplication (explained fully in Section 11). `test` orders have no lines, so these three features stay NaN for them.

In [ ]:
cart = (items_all.groupby("order_id")
        .agg(cart_size=("product_id", "size"), reordered_items=("reordered", "sum"))
        .reset_index())
cart["reorder_rate"] = cart["reordered_items"] / cart["cart_size"]            # arithmetic feature

rows_before = len(orders)
orders = orders.merge(cart, on="order_id", how="left", validate="one_to_one")   # one row per order on both sides
print("rows before merge:", rows_before, "| rows after merge:", len(orders), "(unchanged -> no row multiplication)")

# --- Feature 8 (conditional): cart_size_band ---
orders["cart_size_band"] = pd.cut(orders["cart_size"], bins=[0, 5, 10, 20, np.inf], labels=["1-5 items", "6-10 items", "11-20 items", "21+ items"])
orders[["order_id", "order_number", "cart_size", "reordered_items", "reorder_rate", "cart_size_band"]].head()

### 7.2 Product-level string feature: `is_organic`
A **string operation**: `product_name.str.lower().str.contains("organic")` flags products whose name contains the word "organic". Purpose: lets us later measure how much of a basket is organic and compare departments. Formula uses only the text column; there are no missing names (Section 6), so no special NaN handling is needed. It is a *name-based flag*, so it can miss organic products that do not say so in the name.

In [ ]:
products = raw_products.copy()
products["is_organic"] = products["product_name"].str.lower().str.contains("organic")      # Feature 9 (string operation)
print("organic-named products:", int(products["is_organic"].sum()), f"({100*products['is_organic'].mean():.1f}% of catalogue)")
products[["product_id", "product_name", "is_organic"]].head()

### 7.3 Feature documentation table
For each new feature: formula/source, purpose, data type, missing-value treatment (type and missing counts are read from the data by code).

In [ ]:
feature_doc = pd.DataFrame([
    ("time_of_day",        "pd.cut(order_hour_of_day) into 4 bands",              "Compare night / morning / afternoon / evening behaviour", "orders"),
    ("gap_category",       "pd.cut(days_since_prior_order) into 5 bands",         "Readable view of re-order rhythm incl. the capped 30+ group", "orders"),
    ("is_first_order",     "order_number == 1",                                    "Separate first orders (no history, no re-orders possible)", "orders"),
    ("is_weekend_assumed", "order_dow in {0, 1}  (ASSUMPTION)",                    "Weekend vs weekday contrast, clearly flagged as assumption", "orders"),
    ("cart_size",          "count of order lines per order",                       "Basket size — core measure for Q2", "orders"),
    ("reordered_items",    "sum(reordered) per order",                             "Numerator of the reorder rate", "orders"),
    ("reorder_rate",       "reordered_items / cart_size",                          "Share of basket that is repeat purchases — core measure for Q3/Q4", "orders"),
    ("cart_size_band",     "pd.cut(cart_size) into 4 bands",                       "Group small/medium/large baskets", "orders"),
    ("is_organic",         "product_name.str.lower().str.contains('organic')",     "Organic share analysis", "products"),
], columns=["feature", "formula / source columns", "purpose", "table"])
feature_doc["data type"] = [str((orders if t == "orders" else products)[f].dtype) for f, t in zip(feature_doc["feature"], feature_doc["table"])]
feature_doc["missing values"] = [int((orders if t == "orders" else products)[f].isna().sum()) for f, t in zip(feature_doc["feature"], feature_doc["table"])]
feature_doc["missing treatment"] = ["none missing", "NaN kept for first orders", "none missing", "none missing",
                                    "NaN kept for test orders (no lines)", "NaN kept for test orders", "NaN kept for test orders", "NaN kept for test orders", "none missing"]
feature_doc

**Interpretation.** Nine engineered features: five categorical/logical (`time_of_day`, `gap_category`, `is_first_order`, `is_weekend_assumed`, `cart_size_band`), three numeric (`cart_size`, `reordered_items`, `reorder_rate`) and one string-based (`is_organic`, 5,036 products = 10.1% of the catalogue).
**Observations.** (1) Missing values are only where the source is missing: 20,000 for the gap-based band (first orders) and 7,202 for the three basket features (test orders). (2) The merge that attached the basket features kept 329,109 rows (one-to-one), so no row multiplication happened.
**Decision.** Preserve NaN rather than imputing; each feature has a stated analytical use (Q1–Q4).
**Conclusion.** Task 1 is met with more than four meaningful features, each documented.

# 8. Grouping and Aggregation (Task 2)

**Objective.** Answer *meaningful group questions* (how do basket size and re-ordering differ by day, by time of day and by customer?) using `groupby()`, `agg()` with multiple functions, and `transform()`.

**Concept.**
* **Aggregation** (`groupby().agg()`) *reduces* each group to **one summary row** — the result is shorter than the input.
* **Transformation** (`groupby().transform()`) computes a group-level value but returns it **aligned to every original row** — the result has the same length/index as the input, so it can be used directly in row-level arithmetic (e.g. "this order minus the customer's average").

**Methodology.** Named aggregation (`agg(new_name=("column", "func"))`) for readable output; list-style `agg([...])` to show many functions at once; `transform("mean")` and `transform("size")` for row-aligned group values.

### 8.1 `groupby` + `agg` with multiple functions — by day code

In [ ]:
dow_summary = (orders.groupby("dow_name", observed=True)
    .agg(n_orders=("order_id", "size"),
         mean_cart=("cart_size", "mean"), median_cart=("cart_size", "median"),
         min_cart=("cart_size", "min"), max_cart=("cart_size", "max"),
         total_items=("cart_size", "sum"),
         mean_reorder_rate=("reorder_rate", "mean"),
         mean_gap_days=("days_since_prior_order", "mean")))
dow_summary

**Interpretation.** Orders are concentrated on Day 0 (57,764) and Day 1 (56,613); the lowest volume is Day 4 (40,663). Basket size follows a similar pattern: mean 11.17 items on Day 0 and 10.73 on Day 6, versus 9.26 on Day 3. The mean re-order rate stays between 0.58 and 0.61 on all days, and the mean gap between 10.5 and 11.8 days.
**Observations.** (1) The mean is above the median (e.g. 11.17 vs 10 on Day 0) on every day, a sign of right-skewed baskets (Section 12). (2) The maximum basket is 102 items on Days 2 and 4, so a few very large orders exist on several days. (3) The re-order rate barely changes by day, so day of week is a weak driver of repeat buying.
**Decision.** Use the mean *and* the median together when comparing groups.
**Conclusion.** Day mainly changes *how many* orders arrive and slightly *how big* they are, not how repetitive they are.

In [ ]:
# list-style agg: the same idea with several functions applied to one column, by time of day
tod_summary = orders.groupby("time_of_day", observed=True)["cart_size"].agg(["count", "mean", "median", "std", "min", "max"])
tod_summary

**Interpretation.** Afternoon (152,117 orders with known basket) and morning (105,872) dominate; night has only 6,103. Mean basket size is almost identical across time bands (9.93–10.10), and all medians are 8.
**Observations.** (1) Time of day changes the *number* of orders enormously but not the *size* of the basket. (2) The standard deviation (7.4–7.7) is similar in every band.
**Decision.** Treat time of day as a volume/planning variable rather than a basket-size driver.
**Conclusion.** Delivery capacity should follow order volume by band; basket size does not need a time-of-day adjustment.

### 8.2 Customer-level aggregation (one row per user)
We now change the unit of analysis from *order* to *customer*: `user_df` has one row per sampled user. `n_orders` uses `max(order_number)` so it counts every order including the hidden `test` order; averages use only the orders where the value is known (pandas skips NaN).

In [ ]:
user_df = (orders.groupby("user_id")
    .agg(n_orders=("order_number", "max"),
         mean_cart=("cart_size", "mean"),
         mean_gap_days=("days_since_prior_order", "mean"),
         mean_hour=("order_hour_of_day", "mean"),
         mean_reorder_rate=("reorder_rate", "mean"),
         total_items=("cart_size", "sum")))
print("user_df rows (one per user):", len(user_df), "| orders rows:", len(orders))
user_df.describe().T[["count", "mean", "std", "min", "50%", "max"]]

**Interpretation.** `user_df` has 20,000 rows (one per customer) versus 329,109 order rows. Customers place on average 16.5 orders (median 10, max 100), average 9.9 items per order, wait on average 15.5 days between orders, shop on average around hour 13.6, and re-order on average 46.7% of their basket.
**Observations.** (1) Customer mean orders (16.5) is well above the median (10): a minority of very loyal customers. (2) The customer-level average gap (15.5 days) is larger than the order-level average (11.1 days) because frequent shoppers contribute many short gaps at the order level but only one row each at the customer level — a good example of how the **unit of analysis changes the answer**.
**Decision.** Keep both levels and always state which one a number refers to.
**Conclusion.** Customer-level aggregation gives one row per user, ready for Sections 13–16.

### 8.3 `transform()` — values aligned to the original rows
Each order's **deviation from its own customer's average basket**, plus the customer's number of orders, written back onto every order row.

In [ ]:
by_user = orders.groupby("user_id")
orders["user_mean_cart"]     = by_user["cart_size"].transform("mean")      # same length as orders (transform)
orders["user_order_count"]   = by_user["order_id"].transform("size")
orders["cart_vs_user_mean"]  = orders["cart_size"] - orders["user_mean_cart"]

agg_result = by_user["cart_size"].mean()                                    # agg: one row per user (shorter)
print("aggregation length   :", len(agg_result), "(one row per user)")
print("transformation length:", len(orders["user_mean_cart"]), "(same as orders -> aligned to original rows)")
print("check: deviations average ~0 inside each customer ->", round(float(orders.groupby('user_id')['cart_vs_user_mean'].mean().abs().max()), 10))
orders.loc[orders["user_id"] == orders["user_id"].iloc[0], ["user_id", "order_number", "cart_size", "user_mean_cart", "cart_vs_user_mean"]].head(8)

**Interpretation.** `agg` returned 20,000 rows (one per customer); `transform` returned 329,109 values aligned to the original orders. For customer 1 the average basket is 6.36 items, so the first order (5 items) is −1.36 below that customer's own average.
**Observations.** (1) The check shows deviations average to exactly 0 inside each customer, as they must. (2) Because `transform` keeps the original rows, we can compare each order with its customer's own baseline without a separate merge.
**Decision.** Use `agg` when we want a summary table; use `transform` when a group value is needed in row-level calculations.
**Conclusion.** The difference is demonstrated: aggregation *reduces*, transformation *aligns*.

### 8.4 Group comparison chart
Mean basket size and number of orders by day code (from `dow_summary`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(x=dow_summary.index.astype(str), y=dow_summary["mean_cart"], color="#4C78A8", ax=axes[0])
axes[0].set(title="Mean basket size by day code", xlabel="Day-of-week code (0-6, unlabelled in source)", ylabel="Mean items per order")
for i, v in enumerate(dow_summary["mean_cart"]): axes[0].text(i, v + 0.05, f"{v:.2f}", ha="center", fontsize=9)
sns.barplot(x=dow_summary.index.astype(str), y=dow_summary["n_orders"], color="#F58518", ax=axes[1])
axes[1].set(title="Number of orders by day code (our sample)", xlabel="Day-of-week code (0-6, unlabelled in source)", ylabel="Orders")
plt.tight_layout(); plt.show()

**Observations.** (1) Mean basket size is highest on Day 0 (11.17) and Day 6 (10.73) and lowest on Day 3 (9.26) — a spread of about 1.9 items. (2) Order volume is highest on Days 0 and 1 (≈57.8k and ≈56.6k) and about 30% lower on Day 4 (40,663) — volume varies more than basket size. (3) The busiest days are not the days with the biggest baskets only: Day 1 has high volume but a mid-range basket (10.09).
**Conclusion.** Both volume and basket size peak around codes 0 and 6, supporting a "stock-up around the weekend" reading — as an *association*, given the unlabelled day codes.

# 9. Pivot Tables and Crosstabulation (Task 3)

**Objective.** Summarise a numeric measure across *two* categorical dimensions (`pivot_table`) and count/proportion combinations of two categorical variables (`crosstab`).

**Concept.** A **pivot table** is a two-way group-by: one categorical variable becomes rows, another becomes columns, and a numeric measure is aggregated in each cell. A **crosstab** (contingency table) counts how often each combination of two categories occurs; `normalize` turns counts into proportions and `margins` adds totals.

**Methodology.** `pd.pivot_table(values, index, columns, aggfunc, margins=True)` for mean basket size by day × time of day, and for count + mean together. `pd.crosstab` with `margins=True` and with `normalize="index"` (row percentages).

In [ ]:
pivot_cart = pd.pivot_table(orders, values="cart_size", index="dow_name", columns="time_of_day",
                            aggfunc="mean", margins=True, margins_name="All", observed=True)
pivot_cart

**Interpretation.** The pivot table gives mean basket size for every day × time-of-day cell; the "All" margins reproduce the group means (10.06 overall). The largest cell is Day 0 morning (11.43); the smallest is Day 3 morning (9.12).
**Observations.** (1) Day matters more than time of day: row means range 9.26–11.17, column means only 9.93–10.10. (2) The gap between Day 0 morning and Day 3 morning (2.3 items) is larger than any difference between time bands within one day.
**Conclusion.** Day-of-week is the more informative dimension for basket size.

In [ ]:
pivot_two = pd.pivot_table(orders, values="cart_size", index="is_weekend_assumed", columns="gap_category",
                           aggfunc=["count", "mean"], observed=True)
pivot_two

**Interpretation.** The table shows counts and mean basket sizes for assumed weekend vs weekday orders in each re-order-gap band. Same-day re-orders have the smallest baskets (6.9 weekday / 7.4 weekend), 1–7 days 9.3 / 10.2, and 8–14 days the largest (10.7 / 11.5), after which baskets stay slightly lower (15–29 days: 10.4 / 11.2; 30+ days: 9.7 / 10.5).
**Observations.** (1) Assumed-weekend baskets are bigger than weekday baskets in **every** gap band. (2) Basket size rises with the wait up to about two weeks, which fits a "top-up vs full shop" idea — an association only.
**Conclusion.** Both the day type and the gap since the last order are linked to basket size.

In [ ]:
ct_counts = pd.crosstab(orders["dow_name"], orders["time_of_day"], margins=True, margins_name="Total")
ct_rowpct = (pd.crosstab(orders["dow_name"], orders["time_of_day"], normalize="index") * 100).round(1)
display(ct_counts)
print("Row percentages (each row sums to 100): share of the day's orders placed in each time-of-day band")
ct_rowpct

**Interpretation.** The crosstab counts orders per day × time band (total 329,109). In the row-percentage table, 45–51% of each day's orders are placed in the afternoon, 30–37% in the morning, 16–20% in the evening and only 1.7–2.1% at night.
**Observations.** (1) Day 1 is the most "morning-heavy" (36.9%) and Day 0 the most "afternoon-heavy" (50.9%). (2) The shape is similar every day, so the daily profile is stable.
**Conclusion.** Normalising by row makes days of different volume comparable; the ordering rhythm is the same on every day.

In [ ]:
ct_gap = (pd.crosstab(orders["gap_category"], orders["is_weekend_assumed"], normalize="columns") * 100).round(1)
ct_gap.columns = ["weekday orders (%)", "weekend (assumed) orders (%)"]
print("Distribution of re-order gap categories within weekday vs assumed-weekend orders (columns sum to 100):")
ct_gap

**Interpretation.** Gap categories are distributed similarly for weekday and assumed-weekend orders: about 48.8% / 46.8% of orders occur 1–7 days after the previous one and 11.3% / 11.9% at the 30+ cap.
**Observations.** (1) Weekly re-ordering (1–7 days) is the dominant pattern (about half of all orders with a previous order). (2) Day type barely changes the gap mix (differences ≤ 2 percentage points).
**Conclusion.** The re-order rhythm is a property of the customer more than of the day type.

In [ ]:
plt.figure(figsize=(8.5, 4.8))
sns.heatmap(pivot_cart.drop(index="All", columns="All"), annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={"label": "Mean items per order"})
plt.title("Mean basket size: day code x time of day"); plt.xlabel("Time of day"); plt.ylabel("Day-of-week code")
plt.tight_layout(); plt.show()

**Observations.** (1) The darkest cells are Day 0 morning, Day 0 afternoon and Day 6 morning/afternoon/evening (≈10.6–11.4 items). (2) The lightest cells are Day 3 morning and Day 2/3/4 afternoon (≈9.1–9.4). (3) Colour changes more down the rows (days) than across the columns (time bands), confirming that day matters more.
**Conclusion.** The heatmap shows the pivot table at a glance.

# 10. Data Reshaping (Task 4)

**Objective.** Demonstrate `melt()`, `pivot()`, `stack()` and `unstack()` and show why changing between wide and long layouts is useful.

**Concept — wide vs long.**
* **Wide format** stores several measured variables (or categories) in *separate columns* — easy for a human to read, e.g. one column per time-of-day band.
* **Long (tidy) format** stores the *variable name in one column and its value in another*, one observation per row — what plotting libraries (`seaborn` hue/facets) and `groupby` expect, and what you need when you want to stack tables from different sources.
* `melt` = wide → long; `pivot` = long → wide (index/column pairs must be **unique**); `stack` moves column labels into the row index (wide → long); `unstack` moves an index level into columns (long → wide).

**Methodology.** Start from the real wide table "mean basket size by day × time-of-day" (a `pivot_table` result) and a second wide table with three different measures per day.

### 10.1 `melt()` — wide to long

In [ ]:
wide_cart = pd.pivot_table(orders, values="cart_size", index="dow_name", columns="time_of_day", aggfunc="mean", observed=True)
print("WIDE table shape:", wide_cart.shape, "(7 days x 4 time-of-day columns)")
long_cart = (wide_cart.reset_index()
             .melt(id_vars="dow_name", var_name="time_of_day", value_name="mean_cart_size"))
print("LONG table shape:", long_cart.shape, "(one row per day x time-of-day pair = 7 x 4)")
long_cart.head(8)

**Interpretation.** The wide table (7 × 4) becomes a long table of 28 rows, one per day × time-of-day pair, with columns `dow_name`, `time_of_day` and `mean_cart_size`.
**Observations.** (1) No information was lost (7 × 4 = 28). (2) The old column headers (time bands) are now *values* in a column — the defining feature of long format.
**Conclusion.** `melt` converted wide → long as required.

### 10.2 `pivot()` — long back to wide, and the uniqueness rule
`pivot()` needs each (index, column) pair to appear **once**. We (a) round-trip the long table back to wide and check it equals the original, (b) deliberately pivot raw, un-aggregated order rows — which contain *many* rows per (day, time) pair — to show the error, and (c) fix it by aggregating first.

In [ ]:
back_to_wide = long_cart.pivot(index="dow_name", columns="time_of_day", values="mean_cart_size")
back_to_wide = back_to_wide.reindex(index=wide_cart.index, columns=wide_cart.columns)       # pivot() sorts columns alphabetically; restore the original order
print("round trip identical to original wide table:", bool(np.allclose(back_to_wide.values, wide_cart.values)))

try:                                                   # duplicates -> pivot cannot decide which value to use
    orders.head(2000).pivot(index="dow_name", columns="time_of_day", values="cart_size")
except ValueError as e:
    print("pivot on un-aggregated rows fails ->", e)

fixed = (orders.groupby(["dow_name", "time_of_day"], observed=True)["cart_size"].mean().reset_index()
         .pivot(index="dow_name", columns="time_of_day", values="cart_size"))
print("after aggregating duplicates first, pivot works; shape:", fixed.shape)
back_to_wide

**Interpretation.** `pivot()` rebuilt the wide table and it matches the original exactly (after restoring the column order, because `pivot` sorts columns alphabetically). Pivoting un-aggregated order rows fails with "Index contains duplicate entries, cannot reshape", and works after aggregating duplicates first.
**Observations.** (1) `pivot` does not aggregate — it needs each (index, column) pair once; `pivot_table` is the aggregating version. (2) The error message is exactly the risk the assignment warns about.
**Decision.** Always aggregate (`groupby`/`pivot_table`) before `pivot` when duplicates exist.
**Conclusion.** Round-trip long → wide is verified.

### 10.3 `stack()` and `unstack()` on a MultiIndex
A `groupby` on two keys returns a **MultiIndex Series** (long layout). `unstack()` pivots the *inner* index level into columns; `unstack(level=0)` pivots the *outer* one instead; `stack()` reverses it. We also stack a wide table with three different measures.

In [ ]:
multi = orders.groupby(["dow_name", "time_of_day"], observed=True)["cart_size"].mean()     # MultiIndex Series (long)
print("MultiIndex Series  -> index levels:", multi.index.names, "| length:", len(multi))
unstacked = multi.unstack()                       # inner level (time_of_day) -> columns
print("unstack()          -> shape", unstacked.shape)
unstacked_outer = multi.unstack(level=0)          # outer level (dow_name) -> columns
print("unstack(level=0)   -> shape", unstacked_outer.shape)
restacked = unstacked.stack()                     # back to a MultiIndex Series
print("stack() restores the original series:", bool(np.allclose(restacked.sort_index().values, multi.sort_index().values)))
unstacked_outer.round(2)

**Interpretation.** The MultiIndex Series (28 values) becomes a 7 × 4 table with `unstack()` and a 4 × 7 table with `unstack(level=0)`; `stack()` restores the original Series (check = True).
**Observations.** (1) `unstack` moves an index level into columns, `stack` moves columns into the index — they are inverses. (2) The level chosen decides the orientation of the table (days as rows or as columns).
**Conclusion.** `stack`/`unstack` demonstrated on a real MultiIndex; the three-measure table also stacked from 7 × 3 into 21 long rows.

In [ ]:
measures_wide = dow_summary[["mean_cart", "mean_reorder_rate", "mean_gap_days"]]         # WIDE: 3 measures in 3 columns
measures_stacked = measures_wide.stack()                                                    # LONG Series with (day, measure) index
measures_stacked.index.names = ["dow_name", "measure"]
measures_long = measures_stacked.reset_index(name="value")
print("wide:", measures_wide.shape, "-> stacked/long:", measures_long.shape)
measures_long.head(6)

### 10.4 Why long format helps: one plot call, three measures
With the long table, `seaborn` can draw one panel per measure (`col="measure"`) with its own y-scale. With the wide table we would write three separate plotting blocks.

In [ ]:
g = sns.catplot(data=measures_long, x="dow_name", y="value", col="measure", kind="bar", sharey=False, color="#4C78A8", height=3.8, aspect=1.0)
g.set_axis_labels("Day-of-week code", "Value (units differ by panel)")
g.set_titles("{col_name}")
g.figure.suptitle("Three measures by day code from ONE long-format table", y=1.04)
plt.show()

**Observations.** (1) The long table let one `catplot` call draw three panels with independent y-scales (basket size ≈ 9–11, re-order rate ≈ 0.58–0.61, gap ≈ 10.5–11.8 days). (2) The re-order-rate panel is almost flat while the basket-size panel varies visibly — the same message as Section 8.
**Why reshaping helps:** wide format is easier to read; long format is easier to plot, filter and combine.

# 11. Data Integration (Tasks 5 and 6)

**Objective.** Integrate the related tables safely — `merge()`, `join()` and `concat()`, all four join types, key and uniqueness checks, suffixes, index alignment, unmatched-record checks and row-count validation — and prove that no row multiplication or double counting occurs.

**Concept.** *Primary key* (PK): column(s) unique in a table. *Foreign key* (FK): column in a child table that must exist in the parent's PK. *Relationship*: one-to-one, **one-to-many** (one order → many order lines) or many-to-one (many lines → one product). **Join types**: *inner* keeps keys present in both; *left* keeps all left keys; *right* keeps all right keys; *outer* keeps all keys from both, filling NaN where one side is missing. A **merge indicator** (`indicator=True`) labels each result row `both`, `left_only` or `right_only`, which gives an anti-join style unmatched-record check.

**The multiplication trap.** Joining a parent to a child multiplies each parent row by its number of child rows. Any *parent-level* measure (here `days_since_prior_order`) is then counted once per child row, so sums and averages are inflated/weighted. The safe pattern is to **aggregate the child table to the parent's grain first**, then join.

### 11.1 Integration plan (Task 6)

In [ ]:
integration_plan = pd.DataFrame([
    ("1", "orders  ->  order lines (prior+train)", "order_id", "orders.order_id (PK) -> items.order_id (FK)", "one-to-many", "AGGREGATE lines to order grain, then LEFT join onto orders (validate one_to_one)", "Keep all orders; test orders stay with NaN basket features"),
    ("2", "order lines  ->  products", "product_id", "products.product_id (PK) <- items.product_id (FK)", "many-to-one", "LEFT join products onto lines (validate many_to_one)", "Keep every order line; add aisle/department/organic flag"),
    ("3", "products  ->  aisles", "aisle_id", "aisles.aisle_id (PK) <- products.aisle_id (FK)", "many-to-one", "LEFT join (validate many_to_one)", "Add aisle name; no product may be lost"),
    ("4", "products  ->  departments", "department_id", "departments.department_id (PK) <- products.department_id (FK)", "many-to-one", "LEFT join (validate many_to_one)", "Add department name; no product may be lost"),
    ("5", "prior + train lines", "(same schema)", "disjoint order_id sets expected", "stack (append)", "CONCAT axis=0 (+ integrity check)", "One table of all order lines"),
], columns=["step", "tables", "key column(s)", "PK / FK", "expected relationship", "intended operation", "reason"])
integration_plan

**Interpretation.** The plan fixes, before any merge, the keys, the relationship and the join type for each step. Steps 2–4 are many-to-one lookups; Step 1 is a one-to-many relationship that is handled by **aggregating first**.
**Decision.** Left joins everywhere that the "main" table must keep all its rows, and `validate=` on every merge to make pandas raise an error if the key assumption is wrong.

### 11.2 Key uniqueness and row/key counts *before* merging
Primary keys must be unique on the **parent** side of every `validate="one_to_many"` / `"many_to_one"` merge; the child side may repeat.

In [ ]:
key_unique = pd.DataFrame({
    "rows": [len(orders), len(products), len(raw_aisles), len(raw_departments), len(items_all), len(items_all), len(items_all)],
    "distinct key values": [orders["order_id"].nunique(), products["product_id"].nunique(), raw_aisles["aisle_id"].nunique(), raw_departments["department_id"].nunique(),
                            items_all["order_id"].nunique(), items_all["product_id"].nunique(), len(items_all.drop_duplicates(["order_id", "product_id"]))],
}, index=["orders.order_id (PK)", "products.product_id (PK)", "aisles.aisle_id (PK)", "departments.department_id (PK)",
          "items.order_id (FK, repeats allowed)", "items.product_id (FK, repeats allowed)", "items (order_id, product_id) composite PK"])
key_unique["unique key?"] = key_unique["rows"] == key_unique["distinct key values"]
key_unique

**Interpretation.** Orders (329,109), products (49,688), aisles (134) and departments (21) all have unique keys. In the lines table `order_id` repeats (321,907 distinct orders over 3,237,631 lines) and `product_id` repeats (41,257 distinct products), but the pair (`order_id`, `product_id`) is unique.
**Observations.** (1) The parent keys are unique, so one-to-many and many-to-one validation will pass. (2) 3,237,631 lines / 321,907 orders ≈ 10.06 lines per order — the factor by which a naive join will multiply order rows.
**Conclusion.** Key structure confirmed.

### 11.3 Merge 1 — orders ⟵ aggregated order lines, with unmatched-record check
`order_cart` is the order-grain aggregate of the lines (built in Section 7). We merge with `how="left"`, `validate="one_to_one"` and `indicator=True`, and add the line source (`eval_set`) so that both tables share a column name — the overlap is resolved with `suffixes`.

In [ ]:
order_cart = (items_all.groupby("order_id")
              .agg(eval_set=("eval_set", "first"), cart_size=("product_id", "size"), reordered_items=("reordered", "sum")).reset_index())

m1 = raw_orders[["order_id", "user_id", "eval_set", "order_number"]].merge(
        order_cart, on="order_id", how="left", validate="one_to_one", indicator=True, suffixes=("_orders", "_lines"))

print("orders rows:", len(raw_orders), "| order_cart rows:", len(order_cart), "| merged rows:", len(m1), "(left join keeps every order)")
print("overlapping column 'eval_set' was split by suffixes ->", [c for c in m1.columns if c.startswith("eval_set")])
print("\nmerge indicator counts:"); print(m1["_merge"].value_counts().to_string())
print("\nunmatched orders by their own eval_set:"); print(m1.loc[m1["_merge"] == "left_only", "eval_set_orders"].value_counts().to_string())
agree = (m1.loc[m1["_merge"] == "both", "eval_set_orders"] == m1.loc[m1["_merge"] == "both", "eval_set_lines"]).all()
print("\nfor matched orders, eval_set agrees in both tables:", bool(agree))
m1.head(4)

**Interpretation.** Left-joining the order-grain aggregate onto orders keeps all 329,109 rows. The indicator shows 321,907 `both` and 7,202 `left_only`, with 0 `right_only`; every unmatched order is a `test` order. The overlapping column `eval_set` was split into `eval_set_orders` and `eval_set_lines`, and for matched orders the two agree.
**Observations.** (1) 0 `right_only` = no order lines refer to a missing order. (2) The agreement of the two `eval_set` columns is an extra consistency check between the tables.
**Decision.** Keep unmatched orders (expected: test items withheld).
**Conclusion.** Merge 1 is validated: no rows lost, none multiplied.

### 11.4 The row-multiplication trap — demonstrated with real numbers
We join orders **directly** to the un-aggregated lines (`one_to_many`) and compare a parent-level measure against the correct order-level value.

In [ ]:
order_level = orders.loc[orders["cart_size"].notna(), ["order_id", "days_since_prior_order"]]          # correct grain: one row per order (with lines)
naive = order_level.merge(items_all[["order_id", "product_id"]], on="order_id", how="inner", validate="one_to_many")

trap = pd.DataFrame({
    "correct (order grain)": [len(order_level), order_level["order_id"].nunique(), order_level["days_since_prior_order"].sum(), order_level["days_since_prior_order"].mean()],
    "naive join (line grain)": [len(naive), naive["order_id"].nunique(), naive["days_since_prior_order"].sum(), naive["days_since_prior_order"].mean()],
}, index=["rows", "distinct orders", "sum of days_since_prior_order", "mean days_since_prior_order"])
trap["inflation factor"] = trap["naive join (line grain)"] / trap["correct (order grain)"]
print("rows multiplied by the join:", len(naive), "vs", len(order_level), "orders -> x", round(len(naive) / len(order_level), 2), "(= average basket size)")
trap

**Interpretation.** Joining orders directly to lines turns 321,907 orders into 3,237,631 rows (×10.06 = the average basket size). The total of `days_since_prior_order` is inflated from 3,318,477 to 34,659,154 (×10.44), and its mean shifts from 10.99 to 11.41 days. One example order (2398795) appears 6 times, once per item.
**Observations.** (1) The number of *distinct* orders stays the same, so counting `nunique` would hide the problem, but `len`, `sum` and `mean` are all wrong. (2) The mean is biased (+3.8%) because large baskets get more weight.
**Decision.** Aggregate lines to order grain *before* joining. The check that follows confirms the correct approach: the sum of baskets equals the number of lines (3,237,631).
**Conclusion.** One-to-many row multiplication is demonstrated with real numbers, and avoided in the analysis.

In [ ]:
# One concrete order, before and after the naive join
example_id = order_level["order_id"].iloc[1]
print("Order", example_id, "appears", int((naive["order_id"] == example_id).sum()), "times after the naive join, once per order line:")
naive[naive["order_id"] == example_id].head(6)

In [ ]:
# The safe pattern (aggregate first, then join) preserves totals: sum of baskets must equal the number of order lines
safe_total = int(orders["cart_size"].sum())
print("sum of cart_size over orders :", safe_total)
print("number of order lines        :", len(items_all))
print("totals match (no double counting):", safe_total == len(items_all))

### 11.5 Join types on a small, clearly labelled demonstration subset
> **DEMONSTRATION SUBSET — not the full analysis.** To make inner / left / right / outer differ, we need partial overlap between the two sides. Using a fixed seed we draw 1,500 order ids that have lines; the left table gets the first 1,000, the right table the last 1,000 (500 overlap, 500 left-only, 500 right-only).

In [ ]:
rng_demo = np.random.default_rng(SEED)
pool = rng_demo.choice(order_cart["order_id"].to_numpy(), size=1500, replace=False)
left_ids, right_ids = pool[:1000], pool[500:]

demo_left  = raw_orders.loc[raw_orders["order_id"].isin(left_ids),  ["order_id", "user_id", "order_number"]]
demo_right = order_cart.loc[order_cart["order_id"].isin(right_ids), ["order_id", "cart_size"]]
print("left rows:", len(demo_left), "| right rows:", len(demo_right), "| keys in both:", len(set(demo_left['order_id']) & set(demo_right['order_id'])))

counts, retained = {}, {
    "inner": "only order_ids present in BOTH tables",
    "left":  "all left order_ids; right columns NaN where no match",
    "right": "all right order_ids; left columns NaN where no match",
    "outer": "every order_id from either table; NaN where one side is missing"}
for how in ["inner", "left", "right", "outer"]:
    res = demo_left.merge(demo_right, on="order_id", how=how, validate="one_to_one", indicator=True)
    counts[how] = {"rows": len(res), **res["_merge"].value_counts().reindex(["both", "left_only", "right_only"], fill_value=0).to_dict(), "records retained": retained[how]}
pd.DataFrame(counts).T

**Interpretation.** With 500 shared keys, 500 left-only and 500 right-only: inner = 500 rows, left = 1,000, right = 1,000 and outer = 1,500. The anti-join found 500 left-only orders whose `cart_size` is NaN.
**Observations.** (1) The merge indicator makes each join's retained records countable. (2) Only the outer join keeps every key from both sides.
**Conclusion.** The four join types are explained and shown on a labelled demonstration subset.

In [ ]:
# anti-join style check: which left orders have no match on the right?
anti = demo_left.merge(demo_right, on="order_id", how="left", indicator=True).query("_merge == 'left_only'")
print("left-only (anti-join) orders:", len(anti), "-> their cart_size is NaN:", bool(anti["cart_size"].isna().all()))
anti.head(3)

### 11.6 `join()` and index alignment
`DataFrame.join()` merges on the **index**, not on a column, so both tables must first have `order_id` as index (`set_index`). Both demo tables also share a column name here (`order_number` vs a renamed copy) so we pass `lsuffix`/`rsuffix`. Arithmetic between Series aligns on the index *labels*, not on position — labels missing on one side give NaN.

In [ ]:
left_idx  = demo_left.set_index("order_id")
right_idx = demo_right.set_index("order_id").assign(order_number=lambda d: d.index.map(raw_orders.set_index("order_id")["order_number"]))
joined = left_idx.join(right_idx, how="left", lsuffix="_left", rsuffix="_right")
print("join(how='left') rows:", len(joined), "| columns:", list(joined.columns))
matched = joined["cart_size"].notna()                                         # rows that found a partner on the right
print("matched rows:", int(matched.sum()), "| order_number identical on both sides for matched rows:", bool((joined.loc[matched, "order_number_left"] == joined.loc[matched, "order_number_right"]).all()))

# index alignment on two Series that cover different order ids
s_items    = order_cart.set_index("order_id")["cart_size"].loc[left_ids]            # 1,000 labels
s_reorders = order_cart.set_index("order_id")["reordered_items"].loc[right_ids]     # 1,000 labels (500 shared)
ratio = s_reorders / s_items                                                         # aligned by LABEL -> union of 1,500 labels
print("\nSeries alignment -> result length:", len(ratio), "| computed (non-NaN):", int(ratio.notna().sum()), "| NaN where a label was missing on one side:", int(ratio.isna().sum()))
joined.head(3)

**Interpretation.** `join` (index-based) kept 1,000 rows and produced `order_number_left` / `order_number_right`; for all matched rows they agree. Dividing two Series with different order-id indexes returned 1,500 labels, of which only 500 could be computed — 1,000 are NaN because the label was missing on one side.
**Observations.** (1) pandas aligns by **index label**, not by position. (2) Suffixes (`lsuffix`, `rsuffix`) are required in `join` whenever columns overlap.
**Conclusion.** Index alignment, join suffixes and `join()` are demonstrated.

### 11.7 `concat()` — stacking and side-by-side
(1) **Row-wise** (`axis=0`): `prior` + `train` lines. Using `(order_id, product_id)` as the index and `verify_integrity=True` makes pandas *raise* if any line appears in both files — a real validation that the two files are disjoint. (2) A tiny example shows the error it would raise on a true duplicate. (3) **Column-wise** (`axis=1`) aligns on the index.

In [ ]:
idx_prior = raw_prior.set_index(["order_id", "product_id"])
idx_train = raw_train.set_index(["order_id", "product_id"])
stacked = pd.concat([idx_prior, idx_train], verify_integrity=True)         # raises ValueError if any (order_id, product_id) repeats
print("concat prior+train with verify_integrity=True succeeded -> rows:", len(stacked), "= prior", len(idx_prior), "+ train", len(idx_train))

try:                                                                         # what a failure looks like (prior stacked with itself, small slice)
    pd.concat([idx_prior.head(5), idx_prior.head(5)], verify_integrity=True)
except ValueError as e:
    print("verify_integrity on a true duplicate ->", str(e).split(":")[0])

side_by_side = pd.concat([dow_summary["n_orders"], dow_summary["mean_cart"].rename("mean_cart")], axis=1)    # axis=1 aligns on dow_name
side_by_side

**Interpretation.** `concat` with `verify_integrity=True` succeeded for prior + train: 3,237,631 rows = 3,101,947 + 135,684, which proves no (order, product) line is in both files. The same check raises "Indexes have overlapping values" when a true duplicate is stacked. The axis=1 concat aligned two summary Series on `dow_name`.
**Conclusion.** Row-wise and column-wise `concat` shown, with a real integrity validation.

### 11.8 Merge 2–4 — enrich every order line with product, aisle and department
Each merge is **many-to-one** (many lines → one product, many products → one aisle/department) and validated; the row count must stay equal to the number of lines after every step. `indicator=True` proves there are no unmatched foreign keys.

In [ ]:
steps = []
n0 = len(items_all)
step1 = items_all.merge(products[["product_id", "aisle_id", "department_id", "is_organic"]], on="product_id", how="left", validate="many_to_one", indicator=True)
steps.append(("lines + products", n0, len(step1), int((step1["_merge"] != "both").sum())))
step2 = step1.drop(columns="_merge").merge(raw_aisles, on="aisle_id", how="left", validate="many_to_one", indicator=True)
steps.append(("+ aisles", len(step1), len(step2), int((step2["_merge"] != "both").sum())))
step3 = step2.drop(columns="_merge").merge(raw_departments, on="department_id", how="left", validate="many_to_one", indicator=True)
steps.append(("+ departments", len(step2), len(step3), int((step3["_merge"] != "both").sum())))
items_enriched = step3.drop(columns="_merge")
del step1, step2, step3

merge_log = pd.DataFrame(steps, columns=["merge step", "rows before", "rows after", "unmatched keys"])
merge_log["row count preserved"] = merge_log["rows before"] == merge_log["rows after"]
display(merge_log)
items_enriched.head()

**Interpretation.** Each many-to-one merge kept 3,237,631 rows and had 0 unmatched keys, so every order line now has an aisle, a department and an organic flag.
**Observations.** (1) Row counts are identical before and after every step — no multiplication, no loss. (2) `validate="many_to_one"` would have raised an error had a product, aisle or department key been duplicated.
**Conclusion.** `items_enriched` is the validated integrated table used in Section 11.10 onward.

In [ ]:
# Reverse direction: parents without children (expected? we check, we do not drop)
products_never_ordered = products[~products["product_id"].isin(items_all["product_id"])]
print("products never ordered in our sample:", len(products_never_ordered), f"({100*len(products_never_ordered)/len(products):.1f}% of catalogue) -> expected: we sampled ~10% of users")
print("aisles with no products:", int((~raw_aisles['aisle_id'].isin(products['aisle_id'])).sum()), "| departments with no products:", int((~raw_departments['department_id'].isin(products['department_id'])).sum()))

### 11.9 Integration validation summary (Task 6)
All checks in one table, computed from the objects above.

In [ ]:
validation = pd.DataFrame([
    ("orders (left) vs order-level lines", len(raw_orders), len(m1), "orders preserved", f"{int((m1['_merge']=='left_only').sum()):,} orders unmatched — all eval_set='test'", "EXPECTED: test-order items are withheld by the dataset"),
    ("naive orders x lines (trap)", len(order_level), len(naive), "rows multiply", f"x{len(naive)/len(order_level):.2f}", "AVOID: parent measures are weighted by basket size"),
    ("lines + products + aisles + departments", len(items_all), len(items_enriched), "rows preserved", "0 unmatched keys", "VALID: many-to-one lookups"),
    ("prior + train concat", len(raw_prior) + len(raw_train), len(stacked), "rows add up", "0 overlapping lines", "VALID: files are disjoint"),
], columns=["integration", "rows before", "rows after", "effect", "unmatched / note", "verdict"])
validation

**Interpretation.** The summary collects all checks: unmatched orders (7,202, all `test`) are expected; the naive join multiplies rows ×10.06 and is avoided; the three lookups preserve rows; the two line files are disjoint. (The previous cell also showed that 8,431 products (17.0%) never occur in our sample — expected, because we sampled ~10% of users — and that no aisle or department is empty.)
**Decision / sequence rationale.** Aggregate before joining, join lookups last, validate each step — this order avoids double counting.
**Conclusion.** Integration is correct, validated and documented (Tasks 5 and 6).

### 11.10 Analytical payoff — department-level groupby / agg / transform on the integrated table
Now that lines carry department names, we extend Task 2 with **department** groups: multi-function `agg`, a `transform` that compares each product with its department average, and a bar chart. Products are only ranked when they have at least 30 order lines — a **documented decision** because a re-order ratio computed from a handful of lines is mostly noise.

In [ ]:
dept_summary = (items_enriched.groupby("department")
    .agg(order_lines=("order_id", "size"), orders_containing=("order_id", "nunique"),
         reorder_rate=("reordered", "mean"), mean_cart_position=("add_to_cart_order", "mean"),
         organic_share=("is_organic", "mean"))
    .assign(share_of_lines_pct=lambda d: 100 * d["order_lines"] / d["order_lines"].sum())
    .sort_values("order_lines", ascending=False))
dept_summary

**Interpretation.** Produce (959,594 lines, 29.6%) and dairy & eggs (536,597, 16.6%) together account for 46.2% of all lines. Re-order rates are highest for dairy & eggs (0.667), beverages (0.651) and produce (0.649), and lowest for personal care (0.319), pantry (0.345) and international (0.355). The disguised `missing` department has 7,274 lines (0.23%) and a re-order rate of 0.394. Produce has the highest organic-name share (53.4%).
**Observations.** (1) Fresh staples are both the biggest and the most repeated categories. (2) Rarely repeated categories (personal care, pantry) are bought more occasionally.
**Conclusion.** Department-level aggregation answers Q3: re-orders are concentrated in fresh staples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
plot_df = dept_summary.sort_values("order_lines")
axes[0].barh(plot_df.index, plot_df["order_lines"] / 1000, color="#4C78A8")
axes[0].set(title="Order lines by department (our sample)", xlabel="Order lines (thousands)", ylabel="Department")
colors = ["#E45756" if d == "missing" else "#54A24B" for d in plot_df.index]
axes[1].barh(plot_df.index, plot_df["reorder_rate"], color=colors)
axes[1].axvline(items_enriched["reordered"].mean(), color="k", ls="--", lw=1, label=f"overall = {items_enriched['reordered'].mean():.3f}")
axes[1].set(title="Re-order rate by department", xlabel="Share of lines that are re-orders"); axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

**Observations.** (1) Produce is by far the largest department by order lines (≈960k), followed by dairy & eggs (≈537k). (2) The highest re-order rates (dairy & eggs, beverages, produce) are above the overall rate (0.588, dashed line), while personal care, pantry and international fall well below it. (3) The red "missing" bar is small and should not be interpreted as a real department.
**Conclusion.** Volume and loyalty go together for staples; niche departments are bought less habitually.

In [ ]:
# product-level table; keep products with >= 30 lines; transform = product minus its department average
product_df = (items_enriched.groupby(["product_id", "department"])
    .agg(order_lines=("order_id", "size"), reorder_ratio=("reordered", "mean"), mean_cart_position=("add_to_cart_order", "mean"))
    .reset_index())
product_df = product_df.merge(products[["product_id", "product_name"]], on="product_id", how="left", validate="many_to_one")
product_30 = product_df[product_df["order_lines"] >= 30].copy()
product_30["dept_mean_reorder"]   = product_30.groupby("department")["reorder_ratio"].transform("mean")        # transform: aligned to product rows
product_30["reorder_vs_dept_avg"] = product_30["reorder_ratio"] - product_30["dept_mean_reorder"]
print("products with any line:", len(product_df), "| products with >= 30 lines:", len(product_30))
top10 = product_30.sort_values("order_lines", ascending=False).head(10)
top10[["product_name", "department", "order_lines", "reorder_ratio", "dept_mean_reorder", "reorder_vs_dept_avg"]]

**Interpretation.** Of 41,257 products with at least one line, 11,539 have ≥ 30 lines. The top products by lines are Banana (47,834; re-order ratio 0.845), Bag of Organic Bananas (37,897; 0.830) and Organic Strawberries (27,323; 0.781). For produce, the average of product-level ratios is 0.510, so Banana is +0.334 above its department average.
**Observations.** (1) The produce *average of products* (0.510) is lower than the *line-weighted* produce rate (0.649) in the department table, because popular products are re-ordered much more than rare ones. (2) `transform` placed the department average next to every product without a merge.
**Decision.** The ≥ 30 line rule is kept so that ratios are not driven by tiny counts.
**Conclusion.** The most-bought products are also the most re-ordered.

# 12. Descriptive Statistics (Tasks 7–10)

**Variables.** Three order-level numeric variables with different shapes: `order_hour_of_day` (0–23, roughly symmetric), `days_since_prior_order` (0–30, capped), and `cart_size` (items per order, count data). `order_number` is added where useful. Statistics use all orders for which the variable exists (NaNs are skipped, never filled).

In [ ]:
stat_vars = ["order_hour_of_day", "days_since_prior_order", "cart_size", "order_number"]
x = {v: orders[v].dropna() for v in stat_vars}
print({v: len(s) for v, s in x.items()})

## 12.1 Measures of Central Tendency (Task 7)

**Objective.** Describe the "typical" value of each variable.
**Concept.** *Mean* = sum ÷ count (uses every value, so it is **sensitive to extreme values**). *Median* = middle value after sorting (robust to extremes). *Mode* = most frequent value (works for categories too). If the mean is clearly **above** the median the distribution tends to have a long right tail / high extremes; mean ≈ median suggests symmetry.
**Methodology.** `Series.mean()`, `.median()`, `.mode()`. A variable can have several modes, so we show the first and how many tie. The gap `mean − median` is reported absolutely and as % of the median.

In [ ]:
central = pd.DataFrame({
    "mean":   {v: s.mean() for v, s in x.items()},
    "median": {v: s.median() for v, s in x.items()},
    "mode (first)": {v: s.mode().iloc[0] for v, s in x.items()},
    "number of modes": {v: len(s.mode()) for v, s in x.items()},
})
central["mean - median"] = central["mean"] - central["median"]
central["gap as % of median"] = 100 * central["mean - median"] / central["median"]
display(central)

print("Mode for categorical features (most frequent category):")
for col in ["time_of_day", "dow_name", "gap_category", "cart_size_band"]:
    vc = orders[col].value_counts()
    print(f"  {col:14s} -> {vc.index[0]}  ({100*vc.iloc[0]/vc.sum():.1f}% of non-missing orders)")

**Interpretation.**
| variable | mean | median | mode | mean − median |
|---|---|---|---|---|
| order_hour_of_day | 13.47 | 13 | 10 | +0.47 (3.6%) |
| days_since_prior_order | 11.14 | 7 | 30 | +4.14 (59%) |
| cart_size | 10.06 | 8 | 6 | +2.06 (26%) |
| order_number | 17.14 | 11 | 1 (4 ties) | +6.14 (56%) |

Categorical modes: afternoon (47.3% of orders), Day 0 (17.6%), gap band "1–7 days" (48.1%), basket band "1–5 items" (31.3%).
**Observations.** (1) For `days_since_prior_order`, `cart_size` and `order_number` the mean is far above the median — a right tail pulls the mean up. (2) For the hour the two are close (13.47 vs 13), which suggests symmetry. (3) The mode of the gap (30) is the **cap artefact**, so it is *not* a typical gap; the median (7 days, one week) is the better summary. (4) `order_number` has four modes because every customer has orders 1–4 (min 4 orders), making these values equally frequent.
**Decision.** Report the median alongside the mean for skewed variables.
**Conclusion.** The typical order is ~8 items, placed ~7 days after the previous one, around midday; means overstate both.

## 12.2 Measures of Dispersion (Task 8)

**Objective.** Measure how spread out each variable is.
**Concept.** *Variance* = average squared distance from the mean; *standard deviation* (SD) = square root of variance, back in the **original units**; *range* = max − min (depends only on two values, so one extreme moves it a lot). Variance is in **squared units** (e.g. items²).
**Methodology & convention.** pandas `.var()` and `.std()` use the **sample** formula (divide by n−1, `ddof=1`); we use this because our orders are a sample of all Instacart orders. For comparison we also show the population version (`ddof=0`) for one variable — with this many rows the two are almost identical. `coefficient of variation` = SD ÷ mean allows comparing spread across different units.

In [ ]:
dispersion = pd.DataFrame(index=stat_vars)
dispersion["unit"] = ["hour", "days", "items", "order index"]
dispersion["variance (sample, ddof=1)"] = [x[v].var() for v in stat_vars]
dispersion["variance unit"] = ["hour²", "days²", "items²", "index²"]
dispersion["std dev (sample)"] = [x[v].std() for v in stat_vars]
dispersion["min"] = [x[v].min() for v in stat_vars]
dispersion["max"] = [x[v].max() for v in stat_vars]
dispersion["range (max-min)"] = dispersion["max"] - dispersion["min"]
dispersion["coef. of variation"] = dispersion["std dev (sample)"] / pd.Series({v: x[v].mean() for v in stat_vars})
display(dispersion)
print("Population vs sample variance for cart_size:", round(x['cart_size'].var(ddof=0), 4), "vs", round(x['cart_size'].var(ddof=1), 4))

**Interpretation.** Sample variance / SD: hour 17.94 hour² / 4.24 h; gap 85.07 days² / 9.22 days; basket 56.67 items² / 7.53 items; order number 318.15 / 17.84. Ranges: 23 hours, 30 days, 101 items (1 to 102), 99 orders. Coefficients of variation: 0.31, 0.83, 0.75, 1.04.
**Observations.** (1) Variance is in squared units, which is why the SD (original units) is the one we interpret. (2) The basket range (101) is huge compared with its SD (7.5) — it is determined by a single extreme order. (3) The hour has the smallest *relative* spread (CV 0.31), the order number the largest (1.04). (4) Population and sample variance of `cart_size` are 56.6701 and 56.6703, i.e. the n−1 correction is negligible at n = 321,907.
**Convention stated:** sample formula (`ddof=1`).
**Conclusion.** Baskets and waiting times are very variable; order timing within the day is comparatively regular.

## 12.3 Percentiles, Quantiles and the Five-Number Summary (Task 9)

**Objective.** Describe the distribution through ordered cut points rather than the mean.
**Concept.** The *p-th percentile* is the value below which p% of observations fall; a *quantile* is the same idea on a 0–1 scale (the 0.25 quantile = 25th percentile). The **quartiles** Q1, Q2 (median), Q3 split the ordered data into four parts with ~25% each. The **five-number summary** is min, Q1, median, Q3, max. The **interquartile range** IQR = Q3 − Q1 covers the middle 50% of the data and is robust to extremes.
**Methodology.** `Series.quantile([...])` (linear interpolation) and a cross-check with `numpy.percentile`.

In [ ]:
pct_points = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
percentile_table = pd.DataFrame({v: s.quantile(pct_points) for v, s in x.items()})
percentile_table.index = [f"P{int(p*100)}" for p in pct_points]
display(percentile_table)

five_num = pd.DataFrame({v: [s.min(), s.quantile(.25), s.median(), s.quantile(.75), s.max()] for v, s in x.items()},
                        index=["Minimum", "Q1 (25%)", "Median (50%)", "Q3 (75%)", "Maximum"]).T
five_num["IQR (Q3-Q1)"] = five_num["Q3 (75%)"] - five_num["Q1 (25%)"]
display(five_num)
print("quantile(0.25) == numpy.percentile(25) for cart_size:", bool(np.isclose(x['cart_size'].quantile(.25), np.percentile(x['cart_size'], 25))))

**Interpretation.** Basket size: P10 = 2, Q1 = 5, median = 8, Q3 = 14, P90 = 20, P95 = 25, P99 = 35, max = 102, IQR = 9. Gap: Q1 = 4, median = 7, Q3 = 15, IQR = 11, and P90 = P95 = P99 = 30. Hour: Q1 = 10, median = 13, Q3 = 16, IQR = 6. `quantile(0.25)` equals `numpy.percentile(25)`.
**Observations.** (1) 50% of baskets are between 5 and 14 items; 99% are at most 35 items but the maximum is 102 — a very long right tail. (2) At least 10% of orders have a gap at the cap (P90 = 30). (3) Half of all orders are placed between 10:00 and 16:00.
**Conclusion.** Percentiles describe the skewed variables better than the mean ± SD; quartiles and IQR feed the outlier rule in Section 17.

## 12.4 Skewness and Kurtosis (Task 10)

**Objective.** Quantify the *shape* of each distribution.
**Concept.** **Skewness** measures asymmetry: > 0 = longer **right** tail (positive skew), < 0 = longer **left** tail (negative skew), ≈ 0 = roughly symmetric. **Kurtosis** describes tail weight relative to a normal distribution. **Software convention:** `pandas.Series.kurt()` returns **excess kurtosis** (normal = 0), so positive = heavier tails/more extreme values than normal, negative = lighter tails (flatter). *Pearson* kurtosis = excess + 3 (normal = 3); we show it via `scipy.stats.kurtosis(fisher=False)` to make the convention explicit.
**Methodology.** pandas `.skew()` / `.kurt()` (bias-corrected sample versions) plus `scipy.stats.skew` / `kurtosis` (population versions) as a cross-check. The "label" column uses a common rule of thumb (|skew| < 0.5 ≈ symmetric, 0.5–1 moderate, > 1 strong) — a guide, not a test.

In [ ]:
def skew_label(g):
    a = abs(g); side = "positive (right tail)" if g > 0 else "negative (left tail)"
    return "approximately symmetric" if a < 0.5 else (f"moderate {side}" if a < 1 else f"strong {side}")

shape = pd.DataFrame({
    "skewness (pandas, sample)":  {v: s.skew() for v, s in x.items()},
    "skewness (scipy, population)": {v: stats.skew(s) for v, s in x.items()},
    "excess kurtosis (pandas)":   {v: s.kurt() for v, s in x.items()},
    "Pearson kurtosis (scipy)":   {v: stats.kurtosis(s, fisher=False) for v, s in x.items()},
})
shape["skew label (rule of thumb)"] = shape["skewness (pandas, sample)"].apply(skew_label)
shape["tails vs normal"] = np.where(shape["excess kurtosis (pandas)"] > 0.5, "heavier (more extremes)", np.where(shape["excess kurtosis (pandas)"] < -0.5, "lighter (flatter)", "similar to normal"))
shape

**Interpretation.** Skewness / excess kurtosis: hour −0.084 / −0.018; gap 0.982 / −0.304; basket size 1.565 / 3.984 (Pearson kurtosis 6.98); order number 1.833 / 3.533. pandas and scipy skewness agree to three decimals at this sample size.
**Observations.** (1) The hour is essentially symmetric with normal-like tails. (2) Basket size and order number have strong positive skew **and** heavy tails (excess kurtosis ≈ 3.5–4): a few extreme values. (3) The gap has moderate positive skew with *lighter* tails than normal (negative excess kurtosis) because it is capped at 30.
**Software convention:** pandas `.kurt()` = excess kurtosis (normal = 0); scipy with `fisher=False` = Pearson (normal = 3), and the two differ by exactly 3.
**Implications.** For skewed, heavy-tailed variables use the median/IQR, investigate outliers, and avoid assuming normality for raw values (but means of samples can still be treated as normal — Section 15).

# 13. Distribution Analysis (Task 11)

**Objective.** Look at the *shape* of real variables and identify, **only where the data genuinely show it**, an approximately normal, a positively skewed, a negatively skewed and a bimodal distribution.

**Concept.** *Normal*: symmetric bell, mean ≈ median, skew ≈ 0, excess kurtosis ≈ 0. *Positively skewed*: long right tail, mean > median. *Negatively skewed*: long left tail, mean < median. *Bimodal*: two separate peaks, often because **two sub-populations are mixed** (e.g. weekly shoppers and monthly shoppers).

**Methodology.** Histograms (with a density curve for continuous variables, bar-per-value for discrete ones), mean/median lines, and the skewness/kurtosis numbers from Section 12. Three further variables are built at other grains: `mean_hour` per customer (from `user_df`), `reorder_ratio` per product (from `product_30`, products with ≥ 30 lines) and `order_lines` per product. We do **not** force a variable into a label it does not fit; where the real data has no strong example we say so and add a clearly labelled **simulated** one.

In [ ]:
cand = {
    "order_hour_of_day (orders)":           orders["order_hour_of_day"],
    "mean order hour per customer (users)": user_df["mean_hour"],
    "cart_size (orders)":                   orders["cart_size"].dropna(),
    "order lines per product (products >= 30 lines)": product_30["order_lines"],
    "reorder ratio per product (products >= 30 lines)": product_30["reorder_ratio"],
    "days_since_prior_order (orders)":      orders["days_since_prior_order"].dropna(),
}
shape_overview = pd.DataFrame({k: {"n": len(v), "mean": v.mean(), "median": v.median(), "skewness": v.skew(), "excess kurtosis": v.kurt()} for k, v in cand.items()}).T
shape_overview["n"] = shape_overview["n"].astype(int)
shape_overview["mean - median"] = shape_overview["mean"] - shape_overview["median"]
shape_overview["shape reading"] = shape_overview["skewness"].apply(skew_label)
shape_overview

**Interpretation.** Skewness of six real variables: mean order hour per customer 0.008 (mean 13.60, median 13.58); order hour −0.084; basket size 1.565; order lines per product 22.9 (mean 261 vs median 84); product re-order ratio −0.359; days since prior order 0.982.
**Observations.** (1) Only one variable (re-order ratio per product) is negative, and mildly (−0.359 → falls in the "approximately symmetric" band of the rule of thumb). (2) The product-popularity variable is extremely skewed (excess kurtosis 827). We therefore add a labelled simulation for a clear negative skew.

### 13.1 Approximately normal candidates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(orders["order_hour_of_day"], discrete=True, stat="density", color="#4C78A8", ax=axes[0])
axes[0].axvline(orders["order_hour_of_day"].mean(), color="red", ls="--", label=f"mean = {orders['order_hour_of_day'].mean():.2f}")
axes[0].axvline(orders["order_hour_of_day"].median(), color="green", ls=":", label=f"median = {orders['order_hour_of_day'].median():.0f}")
axes[0].set(title="Order hour of day (all orders)", xlabel="Hour of day (0-23)", ylabel="Density"); axes[0].legend()

sns.histplot(user_df["mean_hour"], bins=40, stat="density", kde=True, color="#54A24B", ax=axes[1])
axes[1].axvline(user_df["mean_hour"].mean(), color="red", ls="--", label=f"mean = {user_df['mean_hour'].mean():.2f}")
axes[1].axvline(user_df["mean_hour"].median(), color="green", ls=":", label=f"median = {user_df['mean_hour'].median():.2f}")
axes[1].set(title="Average order hour per customer (20,000 customers)", xlabel="Customer's mean order hour", ylabel="Density"); axes[1].legend()
plt.tight_layout(); plt.show()

**Observations.** (1) **Order hour (left):** mean 13.47 ≈ median 13 and skewness −0.08, but the shape is a flat plateau from about 10:00 to 16:00 with very few night orders — symmetric but *not* bell-shaped. (2) **Customer mean hour (right):** mean 13.60 ≈ median 13.58, skewness 0.008, excess kurtosis 0.33 — a clear single-peaked bell. (3) The right panel is an *average of many orders*, so its near-normality is itself a Central-Limit-Theorem effect.
**Conclusion.** The customer's mean order hour is the best real, approximately normal variable; it is used for the empirical rule.

### 13.2 Positively skewed distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cs = orders["cart_size"].dropna()
sns.histplot(cs, bins=range(1, 71), stat="density", color="#F58518", ax=axes[0])
axes[0].axvline(cs.mean(), color="red", ls="--", label=f"mean = {cs.mean():.2f}"); axes[0].axvline(cs.median(), color="green", ls=":", label=f"median = {cs.median():.0f}")
axes[0].set(title="Basket size per order (x-axis cut at 70 items)", xlabel="Items in order", ylabel="Density", xlim=(0, 70)); axes[0].legend()

pl = product_30["order_lines"]
sns.histplot(pl, bins=60, color="#B279A2", ax=axes[1])
axes[1].axvline(pl.mean(), color="red", ls="--", label=f"mean = {pl.mean():.0f}"); axes[1].axvline(pl.median(), color="green", ls=":", label=f"median = {pl.median():.0f}")
axes[1].set(title="Order lines per product (products with >= 30 lines)", xlabel="Number of order lines in our sample", ylabel="Number of products"); axes[1].legend()
plt.tight_layout(); plt.show()

**Observations.** (1) **Basket size:** peak near 6 items (the mode) and a long right tail; mean 10.06 > median 8; skewness 1.57. (2) **Order lines per product:** most products are rarely ordered (median 84 lines) while a few staples have tens of thousands (mean 261); skewness 22.9. (3) In both, the mean is pulled to the right of the median, the classic sign of positive skew.
**Possible reasons:** most orders are small top-ups with a few large stock-ups; product popularity follows a "few giants, many niche items" pattern.

### 13.3 Negatively skewed distributions — real data and a labelled simulation
The re-order ratio per product is the only real variable with a negative (left) skew, and it is only *mildly* negative. Because the assignment asks us to understand a clearly negative skew, we add a **SIMULATED** example (Beta(8, 2) — a proportion that piles up near 1). **The simulated values are not dataset observations.**

In [ ]:
rng_sim = np.random.default_rng(SEED)
sim_negative = pd.Series(rng_sim.beta(8, 2, size=20_000))          # SIMULATED, not Instacart data

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
rr = product_30["reorder_ratio"]
sns.histplot(rr, bins=40, stat="density", kde=True, color="#E45756", ax=axes[0])
axes[0].axvline(rr.mean(), color="red", ls="--", label=f"mean = {rr.mean():.3f}"); axes[0].axvline(rr.median(), color="green", ls=":", label=f"median = {rr.median():.3f}")
axes[0].set(title="REAL: re-order ratio per product (>= 30 lines)", xlabel="Share of a product's lines that are re-orders", ylabel="Density"); axes[0].legend()

sns.histplot(sim_negative, bins=40, stat="density", kde=True, color="gray", ax=axes[1])
axes[1].axvline(sim_negative.mean(), color="red", ls="--", label=f"mean = {sim_negative.mean():.3f}"); axes[1].axvline(sim_negative.median(), color="green", ls=":", label=f"median = {sim_negative.median():.3f}")
axes[1].set(title="SIMULATED (not real data): strongly negative skew", xlabel="Simulated proportion", ylabel="Density"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"real re-order ratio skewness: {rr.skew():.3f} | simulated skewness: {sim_negative.skew():.3f}")

**Observations.** (1) **Real:** product re-order ratio has skewness −0.359 and mean 0.516 slightly below median 0.530 — a *mild* left tail of rarely re-bought products. (2) **Simulated:** Beta(8, 2) has skewness −0.834, with a clear pile-up near 1 and a long left tail, mean < median. (3) The real data do **not** contain a strongly negative variable, so the simulation (labelled, grey) is used only to illustrate the shape.
**Conclusion.** Mean < median is the signature of negative skew; the simulated values are not Instacart observations.

### 13.4 Bimodal distribution — days since prior order

In [ ]:
gap_all = orders["days_since_prior_order"].dropna()
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(gap_all, discrete=True, stat="percent", color="#4C78A8", ax=ax)
ax.axvline(gap_all.mean(), color="red", ls="--", label=f"mean = {gap_all.mean():.2f}"); ax.axvline(gap_all.median(), color="green", ls=":", label=f"median = {gap_all.median():.0f}")
ax.set(title="Days since prior order (orders that have a previous order)", xlabel="Days since prior order (30 means '30 or more')", ylabel="Percent of orders"); ax.legend()
plt.tight_layout(); plt.show()

gap_pct = gap_all.value_counts(normalize=True).sort_index() * 100
print("highest bars (day: % of orders):", gap_pct.sort_values(ascending=False).head(4).round(2).to_dict())
print("local peak check -> day 7:", round(gap_pct[7], 2), "| neighbours 6 and 8:", round(gap_pct[6], 2), round(gap_pct[8], 2), "| day 14:", round(gap_pct[14], 2), "vs 13 and 15:", round(gap_pct[13], 2), round(gap_pct[15], 2))

**Observations.** (1) There are two clear peaks: a rise to 7 days (9.75% of orders, vs 7.46% at day 6 and 5.67% at day 8) and a tall bar at 30 (11.52%). (2) Smaller bumps appear at 14 days (3.12% vs 2.62% and 2.09% at 13 and 15), i.e. multiples of a week. (3) The mean (11.1) lies between the peaks, where few orders occur — the mean is not a "typical" value for a bimodal variable.
**Possible subgroups:** weekly shoppers (peaks at 7, 14, 21…) and infrequent or lapsed shoppers whose wait is 30+ days.
**Caution.** The second peak at 30 is partly an **artefact of the cap** (all gaps ≥ 30 days are stored as 30; day 29 has just 0.63%). The bimodal shape is therefore real in terms of mixed behaviour but its right peak is inflated by censoring.

# 14. Empirical Rule — 68–95–99.7 (Task 12)

**Objective.** Test whether the empirical rule holds for an approximately normal real variable and show where it fails.

**Concept.** For a roughly **bell-shaped, single-peaked, symmetric** distribution about **68%** of values lie within mean ± 1 SD, **95%** within ± 2 SD and **99.7%** within ± 3 SD. (The exact normal-curve values are 68.27%, 95.45%, 99.73%.) For skewed, heavy-tailed or bimodal data the assumptions are not met, so the percentages differ.

**Methodology.** Choose the real variable that Section 13 found most nearly normal — **each customer's average order hour** (skewness ≈ 0, excess kurtosis ≈ 0). Compute mean and SD (`ddof=1`), count the share of observations inside each band, and compare with 68/95/99.7 *and* with the other variables. A SIMULATED normal sample (seed 42, 100,000 values) is included purely as a benchmark and is labelled as such.

In [ ]:
def empirical_rule(series, label):
    s = series.dropna(); m, sd = s.mean(), s.std()
    return {"variable": label, "n": len(s), "mean": m, "sd": sd,
            "within 1 SD (%)": 100 * ((s - m).abs() <= 1 * sd).mean(),
            "within 2 SD (%)": 100 * ((s - m).abs() <= 2 * sd).mean(),
            "within 3 SD (%)": 100 * ((s - m).abs() <= 3 * sd).mean()}

rows = [empirical_rule(user_df["mean_hour"], "REAL: mean order hour per customer"),
        empirical_rule(orders["order_hour_of_day"], "REAL: order hour of day"),
        empirical_rule(orders["cart_size"], "REAL: cart_size (positive skew)"),
        empirical_rule(user_df["n_orders"], "REAL: orders per customer (strong positive skew)"),
        empirical_rule(orders["days_since_prior_order"], "REAL: days since prior order (bimodal)"),
        empirical_rule(pd.Series(np.random.default_rng(SEED).normal(size=100_000)), "SIMULATED normal benchmark")]
emp = pd.DataFrame(rows).set_index("variable")
theory = pd.Series({"within 1 SD (%)": 100 * (stats.norm.cdf(1) - stats.norm.cdf(-1)), "within 2 SD (%)": 100 * (stats.norm.cdf(2) - stats.norm.cdf(-2)), "within 3 SD (%)": 100 * (stats.norm.cdf(3) - stats.norm.cdf(-3))})
emp.loc["THEORY: normal curve"] = [np.nan, np.nan, np.nan, *theory.values]
emp["n"] = emp["n"].astype("Int64")
emp

**Interpretation.** Percent of values inside mean ± 1/2/3 SD: customer mean hour **69.5 / 95.3 / 99.5** (theory 68.3 / 95.4 / 99.7; simulated normal 68.4 / 95.4 / 99.7). Order hour 64.0 / 95.1 / 99.3; basket size 74.7 / 95.6 / 98.5; orders per customer 88.0 / 94.7 / 97.6; days since prior order 74.6 / 88.5 / 100.0.
**Observations.** (1) The customer mean hour follows the rule very closely (within ~1.2 points at ±1 SD). (2) Skewed variables put too many values within 1 SD (74.7%, 88.0%) and too few within 3 SD (98.5%, 97.6%) because a few extremes inflate the SD. (3) The bimodal gap puts only 88.5% within 2 SD and exactly 100% within 3 SD (bounded variable). (4) The flat-topped hour has fewer values within 1 SD (64.0%).
**Conclusion.** The empirical rule holds where the distribution is roughly normal and single-peaked, and fails for skewed, heavy-tailed or bimodal data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
s = user_df["mean_hour"]; m, sd = s.mean(), s.std()
sns.histplot(s, bins=50, stat="density", color="#9ecae1", ax=axes[0])
xs = np.linspace(s.min(), s.max(), 400); axes[0].plot(xs, stats.norm.pdf(xs, m, sd), color="k", lw=1.5, label="normal curve (same mean, SD)")
for k, c in zip((1, 2, 3), ("#d62728", "#ff7f0e", "#2ca02c")):
    axes[0].axvline(m - k * sd, color=c, ls="--"); axes[0].axvline(m + k * sd, color=c, ls="--", label=f"±{k} SD: {emp.iloc[0, 2 + k]:.1f}% inside")
axes[0].set(title="Mean order hour per customer with ±1/2/3 SD bands", xlabel="Customer's mean order hour", ylabel="Density"); axes[0].legend(fontsize=8)

cs = orders["cart_size"].dropna(); m2, sd2 = cs.mean(), cs.std()
sns.histplot(cs, bins=range(1, 71), stat="density", color="#fdd0a2", ax=axes[1])
for k, c in zip((1, 2, 3), ("#d62728", "#ff7f0e", "#2ca02c")):
    axes[1].axvline(m2 + k * sd2, color=c, ls="--", label=f"mean + {k} SD = {m2 + k * sd2:.1f}")
axes[1].axvline(m2 - sd2, color="#d62728", ls="--", label=f"mean - 1 SD = {m2 - sd2:.1f} (below 1 item: impossible)")
axes[1].set(title="Counter-example: basket size (skewed), x cut at 70", xlabel="Items in order", ylabel="Density", xlim=(-5, 70)); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Observations.** (1) **Left:** the histogram of customers' mean order hour closely follows the overlaid normal curve; the legend shows 69.5%, 95.3% and 99.5% inside ±1, 2 and 3 SD. (2) **Right:** for basket size, mean − 1 SD = 2.5 items and the right-hand lines (17.6, 25.1, 32.6) cut a long tail, so the symmetric ± SD intervals do not describe the data. (3) The right panel shows why the rule is unsuitable for skewed data: the intervals are symmetric but the data are not.

# 15. Sampling and the Central Limit Theorem (Task 13)

**Objective.** Show that sample means vary from sample to sample, that their distribution becomes more normal and less spread out as the sample size grows, and that this works even when the underlying variable is strongly skewed.

**Concept.**
* **Population / source population:** here, the 321,907 sampled orders that have a known basket size (`cart_size`) — we treat this set as the population we draw from.
* **Sample & sampling variability:** a small random subset; its mean differs from the population mean by chance.
* **Sampling distribution of the mean:** the distribution of sample means over many repeated samples.
* **Central Limit Theorem:** for independent draws with finite variance, the sampling distribution of the mean approaches a normal distribution as n grows, centred at the population mean with standard error σ/√n.

**Methodology.** Simple random sampling **with replacement** (independent draws) using `numpy.random.default_rng(42)`; sample sizes n = 5, 30, 100 and 500; **1,000 repetitions** each; plot the sample-mean histograms against the theoretical normal curve N(μ, σ/√n); compare the observed SD of sample means with σ/√n and report the skewness of the sample means. A second run uses the **bimodal** variable `days_since_prior_order` with n = 30.

In [ ]:
population = orders["cart_size"].dropna().to_numpy()
mu, sigma = population.mean(), population.std(ddof=0)          # population parameters
N_REPS = 1000
sample_sizes = [5, 30, 100, 500]
rng_clt = np.random.default_rng(SEED)

sampling_means = {n: population[rng_clt.integers(0, len(population), size=(N_REPS, n))].mean(axis=1) for n in sample_sizes}

clt_table = pd.DataFrame({
    "sample size n": sample_sizes,
    "repetitions": N_REPS,
    "mean of sample means": [sampling_means[n].mean() for n in sample_sizes],
    "population mean (mu)": mu,
    "SD of sample means (observed)": [sampling_means[n].std(ddof=1) for n in sample_sizes],
    "theoretical SE = sigma/sqrt(n)": [sigma / np.sqrt(n) for n in sample_sizes],
    "skewness of sample means": [stats.skew(sampling_means[n]) for n in sample_sizes],
    "excess kurtosis of sample means": [stats.kurtosis(sampling_means[n]) for n in sample_sizes],
}).set_index("sample size n")
print(f"Population: {len(population):,} orders | mu = {mu:.3f} | sigma = {sigma:.3f} | population skewness = {stats.skew(population):.3f}")
clt_table

**Interpretation.** Population: 321,907 baskets, μ = 10.058, σ = 7.528, skewness 1.565. The mean of the sample means is 10.219 (n = 5), 10.155 (n = 30), 10.059 (n = 100) and 10.059 (n = 500) — close to μ. The observed SD of sample means (3.531, 1.360, 0.731, 0.343) tracks the theoretical σ/√n (3.367, 1.374, 0.753, 0.337). The skewness of the sample means falls from 0.699 (n = 5) to 0.079 (n = 30).
**Observations.** (1) The spread of sample means shrinks as n grows (≈ 1/√n). (2) With n = 5 the sampling distribution still inherits the population's right skew; by n = 30 it is nearly symmetric. (3) With only 1,000 repetitions the skewness of sample means is itself noisy (0.18 at n = 100 vs 0.08 at n = 30), so it need not fall perfectly smoothly.
**Method statement.** Simple random sampling with replacement, seed 42, 1,000 repetitions per size.
**Limitations.** Orders from the same customer are not independent in reality, and the "population" is our 20,000-user sample rather than all of Instacart. Extreme skewness needs larger n for the normal approximation.

In [ ]:
# sampling variability: the first five samples of n = 30 give five different means
rng_demo2 = np.random.default_rng(SEED + 1)
five = [population[rng_demo2.integers(0, len(population), 30)] for _ in range(5)]
pd.DataFrame({"sample": range(1, 6), "sample mean": [s.mean() for s in five], "sample SD": [s.std(ddof=1) for s in five],
              "difference from population mean": [s.mean() - mu for s in five]}).set_index("sample")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
for ax, n in zip(axes.ravel(), sample_sizes):
    se = sigma / np.sqrt(n)
    sns.histplot(sampling_means[n], bins=30, stat="density", color="#4C78A8", ax=ax)
    xs = np.linspace(sampling_means[n].min(), sampling_means[n].max(), 300)
    ax.plot(xs, stats.norm.pdf(xs, mu, se), color="red", lw=1.8, label="normal curve N(mu, sigma/sqrt(n))")
    ax.axvline(mu, color="k", ls="--", lw=1, label=f"population mean = {mu:.2f}")
    ax.set(title=f"Sampling distribution of the mean, n = {n} ({N_REPS} samples)", xlabel="Sample mean basket size (items)", ylabel="Density"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Observations.** (1) n = 5: the histogram is still right-skewed, with a tail up to ~25 items. (2) n = 30: the histogram is already close to the red normal curve and much narrower (roughly 6.5–14.5 items). (3) n = 100 and 500: narrower again (≈ 8–12.4 and 9–11.2) and centred on the population mean (dashed line, 10.06). (4) Sample means vary from sample to sample (the five n = 30 samples above gave means from 7.97 to 10.77), but the *distribution* of means is stable and predictable.
**Conclusion.** The Central Limit Theorem works even though basket size is strongly skewed.

In [ ]:
# second demonstration: a BIMODAL population (days since prior order), n = 30
pop_gap = orders["days_since_prior_order"].dropna().to_numpy()
means_gap = pop_gap[np.random.default_rng(SEED).integers(0, len(pop_gap), size=(N_REPS, 30))].mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sns.histplot(pop_gap, discrete=True, stat="density", color="#F58518", ax=axes[0]); axes[0].set(title="Population: days since prior order (bimodal)", xlabel="Days", ylabel="Density")
sns.histplot(means_gap, bins=30, stat="density", color="#4C78A8", ax=axes[1])
xs = np.linspace(means_gap.min(), means_gap.max(), 300); axes[1].plot(xs, stats.norm.pdf(xs, pop_gap.mean(), pop_gap.std() / np.sqrt(30)), color="red", label="normal curve")
axes[1].set(title="1,000 sample means, n = 30", xlabel="Sample mean (days)", ylabel="Density"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"population skewness {stats.skew(pop_gap):.3f} | skewness of sample means {stats.skew(means_gap):.3f} | observed SE {means_gap.std(ddof=1):.3f} vs theoretical {pop_gap.std()/np.sqrt(30):.3f}")

**Observations.** (1) The population (left) is bimodal with peaks at 7 and 30 days. (2) The 1,000 means of samples of 30 (right) form a single bell-shaped histogram; their skewness is 0.070 compared with 0.982 for the population. (3) The observed standard error (1.698) is close to σ/√30 (1.684).
**Conclusion.** The CLT applies to the sampling distribution of the *mean* even when the population itself is bimodal.

# 16. Correlation Analysis (Task 14)

**Objective.** Measure *association* between numeric order variables — covariance, Pearson and Spearman correlation — visualise it with a heatmap and scatter plots, and interpret it **without causal claims**.

**Concept.**
* **Covariance** — direction of joint movement; its size depends on the units of both variables, so it cannot be compared across pairs.
* **Pearson r** — strength of the **linear** association, between −1 and +1, unit-free.
* **Spearman ρ** — Pearson computed on the **ranks**; captures any **monotonic** association and is robust to outliers and non-linearity. If Pearson and Spearman differ a lot, suspect non-linearity or outliers.
* **Correlation ≠ causation.** A correlation can arise from confounders, reverse relationships, selection effects, or built-in (structural) links between how two variables are calculated.

**Methodology.** Variables: `order_number`, `days_since_prior_order`, `order_hour_of_day`, `cart_size`, `reorder_rate`. **Decision:** `order_dow` is excluded from the numeric correlation because its 0–6 codes are *categorical labels*, not an ordered scale (Day 6 is not "more" than Day 0). pandas uses **pairwise-complete** observations, so first orders (no gap) and test orders (no basket) are skipped only for the pairs that need them.

In [ ]:
corr_vars = ["order_number", "days_since_prior_order", "order_hour_of_day", "cart_size", "reorder_rate"]
cov_matrix      = orders[corr_vars].cov()
pearson_matrix  = orders[corr_vars].corr(method="pearson")
spearman_matrix = orders[corr_vars].corr(method="spearman")
print("Covariance matrix (units differ: depends on each variable's scale)")
display(cov_matrix)
print("Pearson correlation matrix")
display(pearson_matrix)
print("Spearman correlation matrix")
spearman_matrix

**Interpretation.** Pearson r: `order_number`–`reorder_rate` **+0.425**; `order_number`–`days_since_prior_order` **−0.363**; `days_since_prior_order`–`reorder_rate` **−0.193**; `days_since_prior_order`–`cart_size` +0.061; `cart_size`–`reorder_rate` −0.038; all pairs with `order_hour_of_day` are within ±0.032. Spearman ρ for the same pairs: +0.553, −0.392, −0.161, +0.102, −0.112, ≈ ±0.04.
**Observations.** (1) The strongest links involve order history: later orders (higher `order_number`) are closer together and contain more re-ordered items. (2) Covariances are not comparable across pairs (e.g. 318 for order_number with itself vs 0.114 for re-order rate), because they depend on units. (3) Basket size is nearly unrelated to the other variables (|r| ≤ 0.06).

In [ ]:
pairs = [("order_number", "reorder_rate"), ("days_since_prior_order", "reorder_rate"), ("cart_size", "reorder_rate"),
         ("days_since_prior_order", "cart_size"), ("order_number", "cart_size"), ("order_hour_of_day", "cart_size")]
rows = []
for a, b in pairs:
    d = orders[[a, b]].dropna()
    rows.append({"pair": f"{a} vs {b}", "n": len(d), "covariance": d[a].cov(d[b]),
                 "Pearson r": stats.pearsonr(d[a], d[b])[0], "Spearman rho": stats.spearmanr(d[a], d[b])[0]})
pair_table = pd.DataFrame(rows).set_index("pair")
pair_table["Spearman - Pearson"] = pair_table["Spearman rho"] - pair_table["Pearson r"]
pair_table["n"] = pair_table["n"].astype(int)
pair_table

**Interpretation.** For `order_number` vs `reorder_rate`: covariance 2.562, Pearson 0.425, Spearman 0.553 (n = 321,907). For `days_since_prior_order` vs `reorder_rate`: −0.547, −0.193, −0.161. For `cart_size` vs `reorder_rate`: −0.095, −0.038, −0.112.
**Observations.** (1) Spearman is **larger** than Pearson for `order_number` (0.553 vs 0.425): the relationship is monotonic but not linear (the band table below shows re-ordering rising steeply in the first orders and more slowly later). (2) For `cart_size` vs `reorder_rate`, Spearman (−0.112) is about three times Pearson (−0.038): extreme baskets and the bounded re-order rate probably dampen the linear measure. (3) All coefficients are weak-to-moderate; none exceeds 0.55.
**No causal claim.** These are associations: for example, customers who order more often may simply be more habitual — we cannot say that waiting less *causes* more re-orders.

### 16.0 Evidence for a non-linear (but monotonic) link
Why can Spearman exceed Pearson for `order_number` vs `reorder_rate`? We look at the average re-order rate by order-number band: a monotonic but curved pattern would explain it.

In [ ]:
band = pd.cut(orders["order_number"], bins=[0, 1, 2, 4, 9, 19, 39, 100], labels=["1", "2", "3-4", "5-9", "10-19", "20-39", "40+"])
orders.groupby(band, observed=True).agg(orders=("order_id", "size"), mean_reorder_rate=("reorder_rate", "mean"))

**Interpretation.** The mean re-order rate is 0.000 for first orders, 0.288 for second orders, 0.438 for orders 3–4, 0.576 for 5–9, 0.691 for 10–19, 0.771 for 20–39 and 0.820 for orders 40 and above.
**Observations.** (1) The rate rises in every band — a monotonic relationship — but the increase shrinks (+0.29, +0.15, +0.14, +0.12, +0.08, +0.05): the curve is concave, so a straight line (Pearson) fits worse than a rank-based measure (Spearman). (2) The first-order value of exactly 0 is the structural effect examined in 16.2.
**Conclusion.** More history is associated with a larger repeat share — an association, since both customers' habits and the catalogue they learn to use may drive it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, mat, name in zip(axes, [pearson_matrix, spearman_matrix], ["Pearson", "Spearman"]):
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0, square=True, linewidths=.5, cbar_kws={"label": f"{name} correlation"}, ax=ax)
    ax.set_title(f"{name} correlation matrix of order variables")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout(); plt.show()

**Observations.** (1) Both heatmaps show the same structure: a positive block `order_number`–`reorder_rate` (0.43 / 0.55) and negative cells for `days_since_prior_order` with `order_number` (−0.36 / −0.39) and `reorder_rate` (−0.19 / −0.16). (2) `order_hour_of_day` is essentially uncorrelated with everything (|r| ≤ 0.04), and `cart_size` has only faint links. (3) Spearman is stronger for `order_number`–`reorder_rate` and `cart_size`–`reorder_rate`, reinforcing the non-linearity noted above.

### 16.1 Scatter plots
Plotting 300,000+ points hides structure, so each order-level scatter uses a **random subset of 5,000 orders** (seed 42, for display only — the coefficients above use all rows). The third panel moves to the **customer level** (one point per customer) to see whether the pattern survives aggregation.

In [ ]:
rng_plot = np.random.default_rng(SEED)
sc = orders.dropna(subset=["days_since_prior_order", "reorder_rate", "cart_size"]).sample(5000, random_state=SEED)
us = user_df.dropna(subset=["mean_gap_days", "mean_reorder_rate"]).sample(5000, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
sns.regplot(x=sc["days_since_prior_order"] + rng_plot.uniform(-.3, .3, len(sc)), y=sc["reorder_rate"], scatter_kws={"alpha": .25, "s": 8}, line_kws={"color": "red"}, ax=axes[0])
axes[0].set(title="Orders: days since prior order vs re-order rate", xlabel="Days since prior order (jittered)", ylabel="Re-order rate (share of basket)")
sns.regplot(x=sc["cart_size"], y=sc["reorder_rate"], scatter_kws={"alpha": .25, "s": 8, "color": "#54A24B"}, line_kws={"color": "red"}, ax=axes[1])
axes[1].set(title="Orders: basket size vs re-order rate", xlabel="Items in order", ylabel="Re-order rate (share of basket)")
sns.regplot(x=us["mean_gap_days"], y=us["mean_reorder_rate"], scatter_kws={"alpha": .25, "s": 8, "color": "#F58518"}, line_kws={"color": "red"}, ax=axes[2])
axes[2].set(title="Customers: mean gap vs mean re-order rate", xlabel="Customer's mean days between orders", ylabel="Customer's mean re-order rate")
plt.tight_layout(); plt.show()
print("n plotted per panel: 5000 | customer-level Pearson r (all 20,000 customers):",
      round(user_df[["mean_gap_days", "mean_reorder_rate"]].corr().iloc[0, 1], 3), "| Spearman:", round(user_df[["mean_gap_days", "mean_reorder_rate"]].corr(method='spearman').iloc[0, 1], 3))

**Observations.** (1) **Left (orders):** a downward trend between days since the prior order and re-order rate, but a very wide scatter (horizontal bands appear because the rate is a ratio of whole numbers, e.g. 0, 0.5, 1). (2) **Middle:** basket size has almost no relationship with re-order rate (Pearson −0.038); the curved dot patterns are the integer-ratio structure again. (3) **Right (customers):** the same negative pattern is much clearer when averaged per customer — Pearson −0.426 and Spearman −0.431 over all 20,000 customers, compared with −0.193 and −0.161 at order level.
**Interpretation.** Averaging removes order-to-order noise, so the association looks stronger at customer level (an *aggregation effect*); it does not mean individual orders are more tightly linked. Association only, not causation.

### 16.2 Robustness check — remove the structural link
In Section 6 we proved that a customer's **first order always has `reorder_rate = 0`** (nothing to re-order yet). That makes `order_number` and `reorder_rate` partly linked *by construction*. We therefore repeat the key correlations after excluding first orders.

In [ ]:
not_first = orders[~orders["is_first_order"]]
robust = pd.DataFrame({
    "Pearson (all orders)":        [orders["order_number"].corr(orders["reorder_rate"]), orders["days_since_prior_order"].corr(orders["reorder_rate"])],
    "Pearson (first orders excluded)": [not_first["order_number"].corr(not_first["reorder_rate"]), not_first["days_since_prior_order"].corr(not_first["reorder_rate"])],
    "Spearman (all orders)":       [orders["order_number"].corr(orders["reorder_rate"], method="spearman"), orders["days_since_prior_order"].corr(orders["reorder_rate"], method="spearman")],
    "Spearman (first orders excluded)": [not_first["order_number"].corr(not_first["reorder_rate"], method="spearman"), not_first["days_since_prior_order"].corr(not_first["reorder_rate"], method="spearman")],
}, index=["order_number vs reorder_rate", "days_since_prior_order vs reorder_rate"])
robust

**Interpretation.** After excluding first orders (whose re-order rate is 0 by definition), the order_number–reorder_rate correlation falls from 0.425 to 0.369 (Pearson) and from 0.553 to 0.464 (Spearman). The gap–reorder_rate correlations are unchanged (−0.193; −0.161) because first orders have no gap and were never in that calculation.
**Observations.** (1) Part of the headline link is **structural** (first order → rate 0), but a clear positive association remains. (2) Reporting both versions is more honest than reporting only the strongest.
**Conclusion.** Re-ordering is associated with a customer's history even after removing the built-in effect; causation is still not established.

# 17. Outlier Detection and Treatment (Tasks 15–16)

**Objective.** Identify unusually large baskets with two methods, inspect them, and compare **removal, capping and retention** before deciding.

**Variable.** `cart_size` (items per order). It is strongly right-skewed (Section 12), which is exactly the situation where outlier rules behave differently, so it is a good test case.

**Concept.**
* **IQR method:** IQR = Q3 − Q1; lower fence = Q1 − 1.5·IQR; upper fence = Q3 + 1.5·IQR; values outside are *potential* outliers. Based on quartiles, so it is robust to the extremes themselves.
* **Z-score method:** Z = (x − mean)/SD; flag |Z| > 3. The mean and SD are themselves **pulled by outliers**, and the rule is most interpretable for roughly normal data — so on skewed data it flags fewer/different points.
* **Potential outlier ≠ error.** An extreme value may be a genuine rare event.
* **Treatments:** *removal* (drop rows — only if confirmed errors / out of scope), *capping/winsorising* (clip to a limit, rows kept), *retention* (keep as genuine).

**Methodology.** Box plot → IQR and Z-score counts → comparison of the two flagged sets → inspect records → three treatments on **separate copies** (the original column is never changed) → compare statistics.

### 17.1 Box plot

In [ ]:
cs = orders["cart_size"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
sns.boxplot(x=cs, color="#9ecae1", flierprops={"markersize": 2, "alpha": .4}, ax=axes[0])
axes[0].set(title="Basket size per order — box plot (all orders)", xlabel="Items in order")
sns.boxplot(x=cs, color="#fdd0a2", showfliers=False, ax=axes[1])
axes[1].set(title="Same box plot without the points beyond the whiskers", xlabel="Items in order")
plt.tight_layout(); plt.show()

**Observations.** (1) The box spans Q1 = 5 to Q3 = 14 with the median at 8, and the upper whisker ends at 27 items. (2) A dense row of points extends beyond the whisker up to 102 items, but none below it (the lower limit would be negative). (3) The median sits closer to Q1 than to Q3, confirming right skew. (4) Removing the points (right panel) shows how compact the core 50% of baskets is.

### 17.2 IQR and Z-score detection

In [ ]:
q1, q3 = cs.quantile(.25), cs.quantile(.75)
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
z_scores = (cs - cs.mean()) / cs.std()
Z_THRESHOLD = 3

mask_iqr = (cs < lower_fence) | (cs > upper_fence)
mask_z   = z_scores.abs() > Z_THRESHOLD
print(f"Q1 = {q1:.0f}, Q3 = {q3:.0f}, IQR = {iqr:.0f} -> fences: [{lower_fence:.1f}, {upper_fence:.1f}]")
print(f"mean = {cs.mean():.2f}, SD = {cs.std():.2f} -> |Z| > {Z_THRESHOLD} means cart_size > {cs.mean() + Z_THRESHOLD*cs.std():.1f} (lower limit {cs.mean() - Z_THRESHOLD*cs.std():.1f} is below the minimum possible 1)")

detect = pd.DataFrame({
    "flagged orders": [int(mask_iqr.sum()), int(mask_z.sum()), int((mask_iqr & mask_z).sum()), int((mask_iqr & ~mask_z).sum()), int((~mask_iqr & mask_z).sum())],
}, index=["IQR method", f"Z-score method (|Z| > {Z_THRESHOLD})", "flagged by BOTH", "IQR only", "Z-score only"])
detect["% of orders"] = 100 * detect["flagged orders"] / len(cs)
detect

**Interpretation.** IQR = 14 − 5 = 9, giving fences [−8.5, 27.5]; the lower fence is below the minimum possible 1, so only large baskets can be flagged. The IQR rule flags 10,452 orders (3.25%); the Z-score rule (|Z| > 3, i.e. > 32.6 items) flags 4,862 (1.51%). All Z-flagged orders are also IQR-flagged (4,862 in both; 5,590 IQR only; 0 Z only).
**Observations.** (1) The methods differ because the Z-score uses the mean and SD, which the extreme values themselves inflate (SD = 7.53), so its cut-off (32.6) is higher than the IQR fence (27.5). (2) On skewed data the IQR rule is the more sensitive one.
**Conclusion.** Both methods only point to *potential* outliers on the right side; inspection is the next step.

### 17.3 Inspecting the flagged records (context before any decision)
Are the biggest baskets plausible? We look at the largest values, whether one customer or many produce them, and how flagged orders differ from the rest.

In [ ]:
flag_df = orders.loc[cs.index].assign(flag_iqr=mask_iqr, flag_z=mask_z)
print("largest 8 baskets:")
display(flag_df.nlargest(8, "cart_size")[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order", "cart_size", "reorder_rate"]])

print("distinct customers behind IQR-flagged orders:", flag_df.loc[flag_df["flag_iqr"], "user_id"].nunique(), "of", int(flag_df["flag_iqr"].sum()), "flagged orders")
print("customers with at least one IQR-flagged order: ", f"{100*flag_df.loc[flag_df['flag_iqr'], 'user_id'].nunique()/orders['user_id'].nunique():.1f}% of all customers")
ctx = flag_df.groupby("flag_iqr").agg(orders=("order_id", "size"), mean_cart=("cart_size", "mean"), mean_reorder_rate=("reorder_rate", "mean"),
                                      mean_gap_days=("days_since_prior_order", "mean"), share_weekend_assumed=("is_weekend_assumed", "mean"))
ctx.index = ["not flagged", "IQR-flagged"]
ctx

**Interpretation.** The largest baskets are 102 items (two orders), 100 and 98 (the latter is a customer's *first* order). The 10,452 IQR-flagged orders come from 3,177 customers (15.9% of customers). Flagged orders average 34.1 items vs 9.25 for the others, with a similar re-order rate (0.607 vs 0.597), a slightly longer gap (12.6 vs 10.9 days) and a higher share of assumed-weekend orders (39.7% vs 34.6%).
**Observations.** (1) The big baskets are spread over many customers (about one in six), so they are not the work of one faulty account. (2) They look like **stock-up shopping** (longer gap, more on the assumed weekend), not data errors — and Section 6.3 verified that every cart's position sequence is valid. (3) The maximum (102) is plausible for a grocery order.
**Decision.** Do not treat the flagged baskets as errors.

### 17.4 Three treatments on separate copies
* **Retention** — `cart_size` unchanged.
* **Removal** — a copy of the data with IQR-flagged orders dropped (`cart_removed`).
* **Capping** — a new column `cart_capped` = `cart_size` clipped to the **upper IQR fence** (the lower fence is below 1 so nothing is capped at the bottom). We also show a 99th-percentile cap for comparison.

The original column is never modified; each treatment lives in its own object.

In [ ]:
cart_retained = cs.copy()                                    # retention: unchanged
cart_removed  = cs[~mask_iqr].copy()                         # removal: separate copy without IQR-flagged rows
cart_capped   = cs.clip(lower=lower_fence, upper=upper_fence)   # capping at IQR fences (separate Series)
cap99         = cs.quantile(0.99)
cart_capped99 = cs.clip(upper=cap99)                         # capping at the 99th percentile

def summarise(s):
    return {"rows": len(s), "mean": s.mean(), "median": s.median(), "std dev": s.std(), "min": s.min(), "max": s.max(), "range": s.max() - s.min(), "skewness": s.skew(), "excess kurtosis": s.kurt()}

treat = pd.DataFrame({"Retention (original)": summarise(cart_retained),
                      "Removal (IQR-flagged dropped)": summarise(cart_removed),
                      f"Capping at IQR upper fence ({upper_fence:.1f})": summarise(cart_capped),
                      f"Capping at P99 ({cap99:.0f})": summarise(cart_capped99)}).T
treat["rows changed"] = [0, len(cs) - len(cart_removed), int((cs != cart_capped).sum()), int((cs != cart_capped99).sum())]
print("rows removed by removal:", len(cs) - len(cart_removed), f"({100*(len(cs)-len(cart_removed))/len(cs):.2f}%) | values altered by IQR cap:", int((cs != cart_capped).sum()))
treat

**Interpretation.**
| treatment | rows | mean | median | SD | range | skewness |
|---|---|---|---|---|---|---|
| Retention | 321,907 | 10.058 | 8 | 7.528 | 101 | 1.565 |
| Removal (IQR) | 311,455 | 9.251 | 8 | 6.083 | 26 | 0.806 |
| Cap at IQR fence (27.5) | 321,907 | 9.844 | 8 | 6.802 | 26.5 | 0.923 |
| Cap at P99 (35) | 321,907 | 9.989 | 8 | 7.229 | 34 | 1.208 |

Removal drops 10,452 rows (3.25%); IQR capping alters the same 10,452 values; P99 capping alters 3,082.
**Observations.** (1) The median (8) is unchanged by every treatment — it is robust. (2) Removal lowers the mean by 0.81 items and the SD by 19% and *redefines the population* (it deletes real large orders). (3) Capping keeps all rows while shrinking the range from 101 to 26.5. (4) The skewness remains positive after every treatment, so the data are still right-skewed.

In [ ]:
# Does the treatment change a downstream result? Correlation of cart_size with reorder_rate under each treatment
base = orders.loc[cs.index, ["cart_size", "reorder_rate"]].copy()
base["cart_capped"] = cart_capped
res = pd.Series({
    "Retention (all orders)": base["cart_size"].corr(base["reorder_rate"]),
    "Removal (IQR-flagged dropped)": base.loc[~mask_iqr, "cart_size"].corr(base.loc[~mask_iqr, "reorder_rate"]),
    "Capping at IQR upper fence": base["cart_capped"].corr(base["reorder_rate"]),
}, name="Pearson r: cart_size vs reorder_rate").to_frame()
res["Spearman rho"] = [base["cart_size"].corr(base["reorder_rate"], method="spearman"), base.loc[~mask_iqr, "cart_size"].corr(base.loc[~mask_iqr, "reorder_rate"], method="spearman"),
                       base["cart_capped"].corr(base["reorder_rate"], method="spearman")]
res

**Interpretation.** The Pearson correlation between basket size and re-order rate is −0.038 (retention), −0.053 (removal) and −0.043 (capping); the Spearman value is −0.112, −0.117 and −0.112.
**Observations.** (1) The conclusion (a weak negative association) is the same under all treatments. (2) Spearman barely moves because it uses ranks.
**Final decision — Retention as the main analysis.** The extreme baskets are genuine, plausible, spread over many customers and informative about stock-up behaviour, so removing them would bias the picture of customer behaviour. The IQR-capped column is used only as a **sensitivity check** for mean-based comparisons; the original `cart_size` is untouched.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
data = pd.DataFrame({"Retention": cart_retained.reset_index(drop=True), "Capping (IQR fence)": cart_capped.reset_index(drop=True)})
sns.boxplot(data=data, orient="h", palette=["#9ecae1", "#fdd0a2"], flierprops={"markersize": 2, "alpha": .4}, ax=axes[0])
axes[0].set(title="Box plots: original vs capped basket size", xlabel="Items in order")
sns.kdeplot(cart_retained, label="Retention (original)", ax=axes[1], clip=(0, 110))
sns.kdeplot(cart_capped, label=f"Capped at {upper_fence:.1f}", ax=axes[1], clip=(0, 110))
sns.kdeplot(cart_removed, label="Removal (IQR-flagged dropped)", ax=axes[1], clip=(0, 110))
axes[1].set(title="Density of basket size under each treatment", xlabel="Items in order", ylabel="Density"); axes[1].legend()
plt.tight_layout(); plt.show()

**Observations.** (1) The capped box plot has no points beyond its whisker (27.5), while the original has a long cloud of points up to 102 items. (2) In the density plot the three curves coincide up to about 20 items and differ only in the tail. (3) Capping creates a **spike at the cap** (27.5): all 10,452 flagged baskets are piled onto one value, an artificial feature of the capped data. (4) The removal curve has no tail at all and sits slightly higher in the body only because the remaining mass is renormalised.
**Conclusion.** Capping and removal change the extreme end, not the core, but each distorts the tail in its own way, which supports retaining the original values for the main analysis.

# 18. Data Visualisation and Interpretation

**Objective.** Confirm that every mandatory visualisation was produced by notebook code, and add two summary visuals that answer the business questions Q1 and Q3 from the problem statement.

**Methodology.** The checklist is a table of where each mandatory graph lives. The two extra graphs reuse the integrated table and `orders`.

In [ ]:
viz_checklist = pd.DataFrame([
    ("Histogram", "Section 13 (normal, skewed, bimodal), Section 14 (with SD bands)", "yes"),
    ("Box plot", "Section 17.1 and 17.4", "yes"),
    ("Scatter plot", "Section 16.1 (three panels)", "yes"),
    ("Bar / comparison chart", "Section 8.4 (day), Section 11.10 (department), Section 10.4 (long-format bars)", "yes"),
    ("Correlation heatmap", "Section 16 (Pearson and Spearman); Section 9 (pivot heatmap)", "yes"),
    ("Sampling-distribution plot", "Section 15 (four sample sizes + bimodal case)", "yes"),
], columns=["mandatory visualisation", "where in this notebook", "produced by code"])
viz_checklist

In [ ]:
# Extra visual 1 (Q1): when do customers order? orders by day code x hour
hour_dow = pd.crosstab(orders["dow_name"], orders["order_hour_of_day"])
plt.figure(figsize=(14, 4.5))
sns.heatmap(hour_dow, cmap="mako_r", cbar_kws={"label": "Number of orders (our sample)"})
plt.title("Order volume by day code and hour of day"); plt.xlabel("Hour of day (0-23)"); plt.ylabel("Day-of-week code")
plt.tight_layout(); plt.show()
busiest = hour_dow.stack().sort_values(ascending=False).head(3)
print("three busiest (day code, hour) cells:", [(str(d), int(h), int(v)) for (d, h), v in busiest.items()])

**Observations.** (1) The busiest cells are Day 1 at 10:00 (5,353 orders), Day 0 at 13:00 (5,337) and Day 0 at 15:00 (5,322). (2) Activity is very low from about 01:00 to 05:00 on all days and is a broad plateau between about 09:00 and 17:00 on every day. (3) Day 0 and Day 1 are brighter in the middle of the day than other days.
**Answer to Q1.** Customers order mainly between late morning and mid-afternoon, with more orders on Days 0 and 1.

In [ ]:
# Extra visual 2 (Q3): top aisles by volume and their re-order rate (the 'missing' aisle excluded from the naming)
aisle_summary = (items_enriched[items_enriched["aisle"] != "missing"].groupby("aisle")
                 .agg(order_lines=("order_id", "size"), reorder_rate=("reordered", "mean")).sort_values("order_lines", ascending=False))
top_aisles = aisle_summary.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9.5, 5.5))
bars = ax.barh(top_aisles.index, top_aisles["order_lines"] / 1000, color=plt.cm.viridis(top_aisles["reorder_rate"] / top_aisles["reorder_rate"].max()))
for b, rr in zip(bars, top_aisles["reorder_rate"]): ax.text(b.get_width() + 3, b.get_y() + b.get_height() / 2, f"re-order {rr:.0%}", va="center", fontsize=8)
ax.set(title="Top 12 aisles by order lines (label = re-order rate)", xlabel="Order lines (thousands, our sample)", ylabel="Aisle"); ax.set_xlim(0, top_aisles["order_lines"].max() / 1000 * 1.18)
plt.tight_layout(); plt.show()
print("lowest re-order rate among the 12 biggest aisles:", top_aisles["reorder_rate"].idxmin(), round(top_aisles["reorder_rate"].min(), 3), "| highest:", top_aisles["reorder_rate"].idxmax(), round(top_aisles["reorder_rate"].max(), 3))

**Observations.** (1) Among the 12 largest aisles the re-order rate ranges from 0.541 (frozen produce) to 0.780 (milk). (2) All 12 of the biggest aisles have a re-order rate above 0.5.
**Answer to Q3.** At aisle level, re-ordering is highest for milk (0.78) and lower for frozen produce (0.54); at department level it is highest for dairy & eggs, beverages and produce (Section 11.10).

# 19. Findings and Observations

The table below pulls the headline numbers straight from the objects computed earlier, so the written findings that follow can be checked against code output.

In [ ]:
headline = pd.Series({
    "customers in sample (of 206,209)": SAMPLE_USERS,
    "orders in sample": len(orders),
    "orders with known basket (prior+train)": int(orders["cart_size"].notna().sum()),
    "order lines analysed": len(items_enriched),
    "mean basket size (items)": cs.mean(),
    "median basket size (items)": cs.median(),
    "overall re-order rate (share of lines)": items_enriched["reordered"].mean(),
    "share of orders placed in the afternoon (12-17)": (orders["time_of_day"] == "Afternoon (12-17)").mean(),
    "share of orders 1-7 days after the previous one (of orders with a previous order)": (orders.loc[orders["gap_category"].notna(), "gap_category"] == "1-7 days").mean(),
    "share of orders at the 30+ day cap (of orders with a previous order)": (gap_all == 30).mean(),
    "busiest day code by orders": str(dow_summary["n_orders"].idxmax()),
    "department with most lines": dept_summary.index[0],
    "department with highest re-order rate": dept_summary["reorder_rate"].idxmax(),
    "department with lowest re-order rate (excl. 'missing'/'other')": dept_summary.drop(["missing", "other"])["reorder_rate"].idxmin(),
}, name="value").to_frame()
headline

### 19.1 Key findings (all numbers from the cells above)

1. **Sample and data quality.** 20,000 customers → 329,109 orders and 3,237,631 order lines. Data are clean in ranges, duplicates and keys; the real issues are structural: 20,000 missing gaps (first orders), 7,202 test orders without lines, a "missing" category with 1,258 products, and a gap variable capped at 30 days (11.5% of gaps).
2. **Ordering rhythm (Q1).** Orders concentrate between late morning and mid-afternoon (47.3% in the afternoon band) and on Days 0 and 1. The typical wait is **7 days** (median; mean 11.1 is pulled up by long waits); 48.1% of orders come 1–7 days after the previous one. The gap distribution has a weekly peak at 7 and a capped peak at 30.
3. **Baskets (Q2).** The median basket is 8 items (mean 10.06), strongly right-skewed (skewness 1.57). Day matters more than time of day: mean basket 9.3–11.2 items by day, but only 9.9–10.1 by time band. Weekend (assumed) baskets are larger in every gap band.
4. **Re-ordering (Q3).** 58.8% of all order lines are re-orders. Re-order rates are highest in dairy & eggs (0.667), beverages and produce, and lowest in personal care (0.319), pantry and international. Produce + dairy & eggs are 46% of all lines.
5. **Associations (Q4).** Re-order rate is positively associated with order number (Pearson 0.43, Spearman 0.55; 0.37 / 0.46 after removing first orders) and negatively with the gap (−0.19 / −0.16 at order level; −0.43 at customer level). Basket size is nearly unrelated to the other variables. These are associations, not causes.
6. **Outliers (Q5).** IQR flags 3.25% and Z-score 1.51% of baskets; they are genuine stock-up orders, so they are **retained** (capping used only as a sensitivity check).
7. **Statistical demonstrations.** The empirical rule holds for the customer mean hour (69.5 / 95.3 / 99.5%) and fails for skewed/bimodal variables; the CLT turns a skewed (and a bimodal) population into a nearly normal sampling distribution by n = 30, with SD of means close to σ/√n.

# 20. Limitations and Future Scope

**Limitations**
* **Sample, not census:** 20,000 users (≈ 9.7%). Results describe this sample; with a different seed the numbers would change slightly (but patterns should hold).
* **No calendar dates:** only day code, hour and a gap capped at 30 days. Seasonality, trends over time and true weekday names cannot be studied; the weekend flag is an **assumption**.
* **Capped gap:** the 30-day cap (11.5% of gaps) biases the mean gap downward and creates an artificial second peak.
* **Test orders have no items:** 7,202 orders (2.2%) are excluded from basket analysis.
* **Dependence:** orders from the same customer are not independent, which matters for the CLT and for p-value-style reasoning; we therefore avoided hypothesis tests.
* **Associations only:** the correlations cannot show cause and effect; first orders create a built-in link between order number and re-order rate.
* **Name-based flags:** `is_organic` is based on the word "organic" in the product name; case-variant product names (100 extra IDs) may split some product-level counts.
* **Simulated demonstrations:** the strongly negative-skew example (Beta(8, 2)) and the normal benchmark are simulated and labelled; they are not Instacart observations.

**Future scope**
* Use the full dataset (e.g. with chunked processing or Dask/Polars) and compare with this sample.
* Model re-order probability with logistic regression or gradient boosting, using customer history features built in this notebook.
* Customer segmentation (weekly vs infrequent shoppers) to explain the bimodal gap.
* Market-basket analysis (product co-occurrence) and aisle-level recommendation ideas.
* Survival-style methods to handle the right-censored 30-day gap properly.

# 21. Final Conclusion

This project analysed a reproducible 20,000-customer sample of the Instacart grocery data (329,109 orders, 3,237,631 order lines) across all six related tables. **Data quality was high** — no impossible values, duplicates or orphan keys — and the real issues (structural missing gaps, withheld test baskets, a disguised "missing" category, a 30-day cap) were documented and handled by retaining and interpreting rather than deleting.

**Technical work.** Nine engineered features; `groupby/agg/transform`, pivot tables and crosstabs; `melt/pivot/stack/unstack`; validated `merge/join/concat` with a real demonstration that a naive one-to-many join multiplies rows ×10.06 and inflates parent-level totals ×10.44, avoided by aggregating first. Descriptive statistics, percentiles, skewness and kurtosis were interpreted for four variables; normal, skewed, negative-skew (with a labelled simulation) and bimodal shapes were examined; the empirical rule held for customers' mean order hour and failed for skewed or bimodal data; the CLT was shown for a skewed and a bimodal population; correlations were measured (Pearson, Spearman, covariance) and checked for structure; outliers were detected by IQR and Z-score and **retained** after inspection, with capping used only as a sensitivity check.

**Answer to the central question.** Customers typically order about **every week** (median 7 days) between **late morning and mid-afternoon**, buy a **median basket of 8 items** (with a long tail of stock-up orders), and about **59% of items are re-orders**, especially in staples such as dairy & eggs, produce and beverages. Re-ordering is **associated** with a longer order history and shorter waits, while basket size is largely unrelated to the other variables. These findings are associations in a sample, not causal effects.